# Импорт основных библиотек

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
import re
import json
import hashlib
from collections import defaultdict

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)


# Настройка путей к данным

In [21]:
PROJECT_ROOT = Path("SNA").resolve()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR_PARQUET = DATA_DIR / "processed/parquet"
PROCESSED_DIR_CSV = DATA_DIR / "processed/csv"

KAGGLE_WC_DIR = RAW_DIR / "archive"
WC_DB_DIR = RAW_DIR / "worldcup_database"
TEXT_DIR = RAW_DIR / "text"

MATCHES_RAW_PATH = KAGGLE_WC_DIR / "matches_1930_2022.csv"
FIFA_RANK_PATH = KAGGLE_WC_DIR / "fifa_mens_rank.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

MODELING_START_YEAR = 1994
MODELING_END_YEAR = 2022

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MATCHES_RAW_PATH:", MATCHES_RAW_PATH)
print("FIFA_RANK_PATH:", FIFA_RANK_PATH)
print("PROCESSED_DIR_PARQUET:", PROCESSED_DIR_PARQUET)
print("PROCESSED_DIR_CSV:", PROCESSED_DIR_CSV)


PROJECT_ROOT: /Users/gaperov/Documents/University/SNA
MATCHES_RAW_PATH: /Users/gaperov/Documents/University/SNA/data/raw/archive/matches_1930_2022.csv
FIFA_RANK_PATH: /Users/gaperov/Documents/University/SNA/data/raw/archive/fifa_mens_rank.csv
PROCESSED_DIR_PARQUET: /Users/gaperov/Documents/University/SNA/data/processed/parquet
PROCESSED_DIR_CSV: /Users/gaperov/Documents/University/SNA/data/processed/csv


In [9]:
required_files = {
    "World Cup matches": MATCHES_RAW_PATH,
    "FIFA rankings": FIFA_RANK_PATH,
}

for name, path in required_files.items():
    if path.exists():
        print(f"[OK] {name}: {path}")
    else:
        print(f"[MISSING] {name}: {path}")


[OK] World Cup matches: /Users/gaperov/Documents/University/SNA/data/raw/archive/matches_1930_2022.csv
[OK] FIFA rankings: /Users/gaperov/Documents/University/SNA/data/raw/archive/fifa_mens_rank.csv


# Безопасное сохранение таблиц

In [26]:
def save_table(df: pd.DataFrame, path: str | Path, index: bool = False) -> None:
    """
    Безопасное сохранение таблицы:
    1. Если путь .parquet — пытается сохранить parquet.
    2. Если parquet не сработал — сохраняет CSV с тем же именем.
    3. Если путь .csv — сохраняет CSV.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    suffix = path.suffix.lower()

    if suffix == ".parquet":
        try:
            df.to_parquet(path, index=index)
            print(f"[OK] Saved parquet: {path}")
            return
        except Exception as e:
            fallback = path.with_suffix(".csv")
            print(f"[WARN] Could not save parquet: {e}")
            print(f"[WARN] Falling back to CSV: {fallback}")
            df.to_csv(fallback, index=index)
            return

    if suffix == ".csv":
        df.to_csv(path, index=index)
        print(f"[OK] Saved csv: {path}")
        return

    raise ValueError(f"Unsupported file extension: {suffix}")


# Стандартизация названий колонок

Что делает: Приводит названия колонок к единому виду:

In [10]:
def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Приводит названия колонок к snake_case:
    - lower case
    - trim spaces
    - пробелы, дефисы, точки -> underscore
    """
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
        .str.replace(".", "_", regex=False)
    )
    return df


# Гибкий выбор колонок

Что делает: Позволяет искать нужную колонку среди нескольких возможных вариантов. Это важно, потому что Kaggle-датасеты могут иметь разные названия колонок.

In [11]:
def pick_first_existing(df: pd.DataFrame, candidates: list[str], required: bool = True):
    """
    Возвращает первую колонку из candidates, которая есть в df.
    Если required=True и ничего не найдено — выбрасывает понятную ошибку.
    """
    cols = set(df.columns)

    for c in candidates:
        if c in cols:
            return c

    if required:
        raise KeyError(
            f"None of candidate columns found: {candidates}. "
            f"Available columns: {list(df.columns)}"
        )

    return None


def require_column(df: pd.DataFrame, candidates: list[str]) -> str:
    """
    Обязательный выбор колонки.
    """
    return pick_first_existing(df, candidates, required=True)


# Словарь алиасов команд

Что делает: Задает ручные соответствия разных написаний сборных. Это нужно для корректного join матчей с FIFA rankings.

In [12]:
TEAM_ALIASES = {
    "usa": "united states",
    "u s a": "united states",
    "u.s.a.": "united states",
    "united states of america": "united states",
    "united states": "united states",

    "korea republic": "south korea",
    "korea republic of": "south korea",
    "south korea": "south korea",
    "republic of korea": "south korea",

    "ir iran": "iran",
    "iran": "iran",

    "germany fr": "west germany",
    "fr germany": "west germany",
    "west germany": "west germany",
    "germany": "germany",

    "soviet union": "soviet union",
    "ussr": "soviet union",
    "russia": "russia",

    "czech republic": "czechia",
    "czechia": "czechia",
    "czechoslovakia": "czechoslovakia",

    "serbia and montenegro": "serbia and montenegro",
    "yugoslavia": "yugoslavia",

    "republic of ireland": "ireland",
    "ireland": "ireland",

    "china pr": "china",
    "china": "china",

    "côte d'ivoire": "ivory coast",
    "cote divoire": "ivory coast",
    "ivory coast": "ivory coast",

    "cape verde islands": "cape verde",
    "cape verde": "cape verde",

    "bosnia and herzegovina": "bosnia and herzegovina",
    "bosnia herzegovina": "bosnia and herzegovina",
}


# Функции нормализации текстов и команд

Что делает: Создает функции:
normalize_text_name — общая нормализация текста;
normalize_team_name — нормализация названий сборных с учетом алиасов.

In [14]:
def normalize_text_name(x) -> str | None:
    """
    Общая нормализация строк:
    - lower
    - trim
    - удаление лишних символов
    - сжатие пробелов
    """
    if pd.isna(x):
        return None

    x = str(x).strip().lower()
    x = x.replace("&", "and")
    x = re.sub(r"\s+", " ", x)
    x = re.sub(r"[^\w\s]", "", x)
    x = re.sub(r"\s+", " ", x).strip()

    return x


def normalize_team_name(x) -> str | None:
    """
    Нормализует название команды и применяет словарь алиасов.
    """
    x = normalize_text_name(x)

    if x is None:
        return None

    return TEAM_ALIASES.get(x, x)


# Функция генерации стабильных ID

Что делает: Создает стабильные хешированные ID для матчей, команд, стадионов, судей, узлов и ребер графа.

In [15]:
def make_hash_id(prefix: str, *parts, length: int = 12) -> str:
    """
    Создает стабильный ID на основе prefix и набора значений.
    """
    raw = "||".join("" if p is None else str(p) for p in parts)
    h = hashlib.md5(raw.encode("utf-8")).hexdigest()[:length]
    return f"{prefix}_{h}"


# Предпросмотр raw-файла матчей

Что делает: Загружает несколько строк raw-файла и показывает реальные названия колонок. Полезно перед запуском гибкого загрузчика.

In [23]:
matches_preview = pd.read_csv(MATCHES_RAW_PATH, nrows=5)
print("Raw columns:")
print(list(matches_preview.columns))

display(matches_preview.head())


Raw columns:
['home_team', 'away_team', 'home_score', 'home_xg', 'home_penalty', 'away_score', 'away_xg', 'away_penalty', 'home_manager', 'home_captain', 'away_manager', 'away_captain', 'Attendance', 'Venue', 'Officials', 'Round', 'Date', 'Score', 'Referee', 'Notes', 'Host', 'Year', 'home_goal', 'away_goal', 'home_goal_long', 'away_goal_long', 'home_own_goal', 'away_own_goal', 'home_penalty_goal', 'away_penalty_goal', 'home_penalty_miss_long', 'away_penalty_miss_long', 'home_penalty_shootout_goal_long', 'away_penalty_shootout_goal_long', 'home_penalty_shootout_miss_long', 'away_penalty_shootout_miss_long', 'home_red_card', 'away_red_card', 'home_yellow_red_card', 'away_yellow_red_card', 'home_yellow_card_long', 'away_yellow_card_long', 'home_substitute_in_long', 'away_substitute_in_long']


,home_team,away_team,home_score,home_xg,home_penalty,away_score,away_xg,away_penalty,home_manager,home_captain,away_manager,away_captain,Attendance,Venue,Officials,Round,Date,Score,Referee,Notes,Host,Year,home_goal,away_goal,home_goal_long,away_goal_long,home_own_goal,away_own_goal,home_penalty_goal,away_penalty_goal,home_penalty_miss_long,away_penalty_miss_long,home_penalty_shootout_goal_long,away_penalty_shootout_goal_long,home_penalty_shootout_miss_long,away_penalty_shootout_miss_long,home_red_card,away_red_card,home_yellow_red_card,away_yellow_red_card,home_yellow_card_long,away_yellow_card_long,home_substitute_in_long,away_substitute_in_long
0,Argentina,France,3,3.3,4.0,3,2.2,2.0,Lionel Scaloni,Lionel Messi,Didier Deschamps,Hugo Lloris,88966,"Lusail Iconic Stadium, Lusail",Szymon Marciniak (Referee) · Paweł Sokolnicki ...,Final,2022-12-18,(4) 3–3 (2),Szymon Marciniak,Argentina won on penalty kicks following extra...,Qatar,2022,Ángel Di María · 36|Lionel Messi · 108,Kylian Mbappé · 81,['36&rsquor;|2:0|Ángel Di María|Assist:|Alexis...,['81&rsquor;|2:2|Kylian Mbappé|Assist:|Marcus ...,NaN,NaN,Lionel Messi (P) · 23,Kylian Mbappé (P) · 80|Kylian Mbappé (P) · 118,NaN,NaN,"['2|1:1|Lionel Messi', '4|2:1|Paulo Dybala', '...","['1|0:1|Kylian Mbappé', '7|3:2|Randal Kolo Mua...",NaN,"['3|1:1|Kingsley Coman', '5|2:1|Aurélien Tchou...",NaN,NaN,NaN,NaN,"['45+7&rsquor;|2:0|Enzo Fernández', '90+8&rsqu...","['55&rsquor;|2:0|Adrien Rabiot', '87&rsquor;|2...",['64&rsquor;|2:0|Marcos Acuña|for Ángel Di Mar...,['41&rsquor;|2:0|Randal Kolo Muani|for Ousmane...
1,Croatia,Morocco,2,0.7,NaN,1,1.2,NaN,Zlatko Dalić,Luka Modrić,Hoalid Regragui,Hakim Ziyech,44137,"Khalifa International Stadium, Doha",Abdulrahman Ibrahim Al Jassim (Referee) · Tale...,Third-place match,2022-12-17,2–1,Abdulrahman Ibrahim Al Jassim,NaN,Qatar,2022,Joško Gvardiol · 7|Mislav Oršić · 42,Achraf Dari · 9,['7&rsquor;|1:0|Joško Gvardiol|Assist:|Ivan Pe...,['9&rsquor;|1:1|Achraf Dari'],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"['69&rsquor;|2:1|Azzedine Ounahi', '84&rsquor;...",['61&rsquor;|2:1|Nikola Vlašić|for Andrej Kram...,['46&rsquor;|2:1|Ilias Chair|for Abdelhamid Sa...
2,France,Morocco,2,2.0,NaN,0,0.9,NaN,Didier Deschamps,Hugo Lloris,Hoalid Regragui,Romain Saïss,68294,"Al Bayt Stadium, Al Khor",César Arturo Ramos (Referee) · Alberto Morín (...,Semi-finals,2022-12-14,2–0,César Arturo Ramos,NaN,Qatar,2022,Theo Hernández · 5|Randal Kolo Muani · 79,NaN,"['5&rsquor;|1:0|Theo Hernández', '79&rsquor;|2...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,['27&rsquor;|1:0|Sofiane Boufal'],['65&rsquor;|1:0|Marcus Thuram|for Olivier Gir...,['21&rsquor;|1:0|Selim Amallah|for Romain Saïs...
3,Argentina,Croatia,3,2.3,NaN,0,0.5,NaN,Lionel Scaloni,Lionel Messi,Zlatko Dalić,Luka Modrić,88966,"Lusail Iconic Stadium, Lusail",Daniele Orsato (Referee) · Ciro Carbone (AR1) ...,Semi-finals,2022-12-13,3–0,Daniele Orsato,NaN,Qatar,2022,Julián Álvarez · 39|Julián Álvarez · 69,NaN,"['39&rsquor;|2:0|Julián Álvarez', '69&rsquor;|...",NaN,NaN,NaN,Lionel Messi (P) · 34,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"['68&rsquor;|2:0|Cristian Romero', '71&rsquor;...","['32&rsquor;|0:0|Mateo Kovačić', '32&rsquor;|0...",['62&rsquor;|2:0|Lisandro Martínez|for Leandro...,"['46&rsquor;|2:0|Mislav Oršić|for Borna Sosa',..."
4,Morocco,Portugal,1,1.4,NaN,0,0.9,NaN,Hoalid Regragui,Romain Saïss,Fernando Santos,Pepe,44198,"Al Thumama Stadium, ath-Thumāma",Facundo Tello (Referee) · Ezequiel Brailovsky ...,Quarter-finals,2022-12-10,1–0,Facundo Tello,NaN,Qatar,2022,Youssef En-Nesyri · 42,NaN,['42&rsquor;|1:0|Youssef En-Nesyri|Assist:|Yah...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Walid Cheddira · 90+3,NaN,"['70&rsquor;|1:0|Achraf Dari', '90+1&rsquor;|1...",['87&rsquor;|1:0|Vitinha'],['57&rsquor;|1:0|Achraf Dari|for Romain Saïss'...,['51&rsquor;|1:0|João Cancelo|for Raphaël Guer...


# Функция загрузки Kaggle World Cup matches

Что делает: 
- Гибко загружает файл матчей ЧМ 1930–2022:
- стандартизирует колонки;
- ищет разные варианты названий;
- нормализует команды;
- создает match_id;
- создает team_id;
- создает target;
- сохраняет только нужные стандартизированные поля.

In [24]:
def load_kaggle_worldcup_matches(path) -> pd.DataFrame:
    """
    Гибкая загрузка historical World Cup matches.
    Поддерживает разные варианты названий колонок в Kaggle-датасетах.
    """
    df = pd.read_csv(path)
    df = standardize_columns(df)

    print("[INFO] Available match columns:")
    print(list(df.columns))

    year_col = pick_first_existing(
        df,
        ["year", "tournament_year", "season", "world_cup_year"],
        required=True,
    )

    date_col = pick_first_existing(
        df,
        ["date", "match_date", "datetime", "utc_date"],
        required=False,
    )

    stage_col = pick_first_existing(
        df,
        ["stage", "round", "match_stage", "round_name"],
        required=False,
    )

    home_col = pick_first_existing(
        df,
        ["home_team", "home_team_name", "team1", "home", "home_team_name_long"],
        required=True,
    )

    away_col = pick_first_existing(
        df,
        ["away_team", "away_team_name", "team2", "away", "away_team_name_long"],
        required=True,
    )

    home_score_col = pick_first_existing(
        df,
        ["home_score", "home_team_goals", "home_goals", "score_home", "home_team_score"],
        required=True,
    )

    away_score_col = pick_first_existing(
        df,
        ["away_score", "away_team_goals", "away_goals", "score_away", "away_team_score"],
        required=True,
    )

    stadium_col = pick_first_existing(
        df,
        ["stadium", "venue", "stadium_name"],
        required=False,
    )

    city_col = pick_first_existing(
        df,
        ["city", "host"],
        required=False,
    )

    referee_col = pick_first_existing(
        df,
        ["referee", "official", "officials", "referee_name"],
        required=False,
    )

    attendance_col = pick_first_existing(
        df,
        ["attendance", "attendence"],
        required=False,
    )

    home_manager_col = pick_first_existing(
        df,
        ["home_manager", "home_team_manager", "home_coach"],
        required=False,
    )

    away_manager_col = pick_first_existing(
        df,
        ["away_manager", "away_team_manager", "away_coach"],
        required=False,
    )

    out = pd.DataFrame()

    out["tournament_year"] = pd.to_numeric(df[year_col], errors="coerce").astype("Int64")

    if date_col:
        out["match_date"] = pd.to_datetime(df[date_col], errors="coerce")
    else:
        out["match_date"] = pd.NaT

    out["stage"] = df[stage_col] if stage_col else None

    out["home_team_name"] = df[home_col]
    out["away_team_name"] = df[away_col]

    out["home_team_norm"] = out["home_team_name"].apply(normalize_team_name)
    out["away_team_norm"] = out["away_team_name"].apply(normalize_team_name)

    out["home_score"] = pd.to_numeric(df[home_score_col], errors="coerce")
    out["away_score"] = pd.to_numeric(df[away_score_col], errors="coerce")

    out["stadium_name"] = df[stadium_col] if stadium_col else None
    out["city"] = df[city_col] if city_col else None
    out["referee_name"] = df[referee_col] if referee_col else None

    if attendance_col:
        out["attendance"] = pd.to_numeric(df[attendance_col], errors="coerce")
    else:
        out["attendance"] = np.nan

    out["home_manager_name"] = df[home_manager_col] if home_manager_col else None
    out["away_manager_name"] = df[away_manager_col] if away_manager_col else None

    out["tournament_id"] = out["tournament_year"].apply(
        lambda y: f"wc_{int(y)}" if pd.notna(y) else None
    )

    out["match_id"] = out.apply(
        lambda r: make_hash_id(
            "match",
            r["tournament_year"],
            r["match_date"],
            r["home_team_norm"],
            r["away_team_norm"],
            r["stage"],
        ),
        axis=1,
    )

    out["home_team_id"] = out["home_team_norm"].apply(
        lambda x: make_hash_id("team", x)
    )

    out["away_team_id"] = out["away_team_norm"].apply(
        lambda x: make_hash_id("team", x)
    )

    out["stadium_id"] = out["stadium_name"].apply(
        lambda x: make_hash_id("stadium", normalize_text_name(x)) if pd.notna(x) else None
    )

    out["referee_id"] = out["referee_name"].apply(
        lambda x: make_hash_id("referee", normalize_text_name(x)) if pd.notna(x) else None
    )

    out["home_manager_id"] = out["home_manager_name"].apply(
        lambda x: make_hash_id("manager", normalize_text_name(x)) if pd.notna(x) else None
    )

    out["away_manager_id"] = out["away_manager_name"].apply(
        lambda x: make_hash_id("manager", normalize_text_name(x)) if pd.notna(x) else None
    )

    def result(row):
        if pd.isna(row["home_score"]) or pd.isna(row["away_score"]):
            return None
        if row["home_score"] > row["away_score"]:
            return "home_win"
        if row["home_score"] < row["away_score"]:
            return "away_win"
        return "draw"

    out["result"] = out.apply(result, axis=1)

    out["target"] = out["result"].map(
        {
            "home_win": 0,
            "draw": 1,
            "away_win": 2,
        }
    )

    out = out.sort_values(
        ["tournament_year", "match_date", "match_id"],
        na_position="last"
    ).reset_index(drop=True)

    return out


# Загрузка и сохранение очищенных матчей

Что делает: 
- Применяет загрузчик к raw-файлу матчей и сохраняет очищенную таблицу.

In [29]:
matches = load_kaggle_worldcup_matches(MATCHES_RAW_PATH)

print(matches.shape)
display(matches.head())
display(matches.tail())

save_table(matches, PROCESSED_DIR_PARQUET / "worldcup_matches_clean.parquet")
save_table(matches, PROCESSED_DIR_CSV / "worldcup_matches_clean.csv")


[INFO] Available match columns:
['home_team', 'away_team', 'home_score', 'home_xg', 'home_penalty', 'away_score', 'away_xg', 'away_penalty', 'home_manager', 'home_captain', 'away_manager', 'away_captain', 'attendance', 'venue', 'officials', 'round', 'date', 'score', 'referee', 'notes', 'host', 'year', 'home_goal', 'away_goal', 'home_goal_long', 'away_goal_long', 'home_own_goal', 'away_own_goal', 'home_penalty_goal', 'away_penalty_goal', 'home_penalty_miss_long', 'away_penalty_miss_long', 'home_penalty_shootout_goal_long', 'away_penalty_shootout_goal_long', 'home_penalty_shootout_miss_long', 'away_penalty_shootout_miss_long', 'home_red_card', 'away_red_card', 'home_yellow_red_card', 'away_yellow_red_card', 'home_yellow_card_long', 'away_yellow_card_long', 'home_substitute_in_long', 'away_substitute_in_long']
(964, 25)


,tournament_year,match_date,stage,home_team_name,away_team_name,home_team_norm,away_team_norm,home_score,away_score,stadium_name,city,referee_name,attendance,home_manager_name,away_manager_name,tournament_id,match_id,home_team_id,away_team_id,stadium_id,referee_id,home_manager_id,away_manager_id,result,target
0,1930,1930-07-13,Group stage,France,Mexico,france,mexico,4,1,"Pocitos, Montevideo",Uruguay,Domingo Lombardi,4444,Raoul Caudron,Juan Luque,wc_1930,match_008af429d7d1,team_e165d4f2174b,team_4edfc924721a,stadium_910df19d9a18,referee_38c6c92bf039,manager_4e437e3fee0c,manager_bfd0d93fef9f,home_win,0
1,1930,1930-07-13,Group stage,United States,Belgium,united states,belgium,3,0,"Parque Central, Montevideo",Uruguay,Jose Macias,18346,Bob Millar,Hector Goetinck,wc_1930,match_1639b517874f,team_152649df347e,team_73fa01094ea6,stadium_7d92fbf7c709,referee_609b04cd8ef3,manager_1d1efef13b70,manager_be7e1c21c043,home_win,0
2,1930,1930-07-14,Group stage,Yugoslavia,Brazil,yugoslavia,brazil,2,1,"Parque Central, Montevideo",Uruguay,Anibal Tejada,24059,Bosko Simonovic,Pindaro De Carvalho,wc_1930,match_1f0a82595fdf,team_950ebd5e5210,team_6e5fa4d9c48c,stadium_7d92fbf7c709,referee_407514960fba,manager_50a77bcfc752,manager_1126c9d060a4,home_win,0
3,1930,1930-07-14,Group stage,Romania,Peru,romania,peru,3,1,"Pocitos, Montevideo",Uruguay,Alberto Warnken,2549,Octav Luchide,Francisco Bru,wc_1930,match_aff97a035956,team_89055f975d7d,team_32e8419a7ecb,stadium_910df19d9a18,referee_2e5451280212,manager_58ce448ba651,manager_1b6d188b83fe,home_win,0
4,1930,1930-07-15,Group stage,Argentina,France,argentina,france,1,0,"Parque Central, Montevideo",Uruguay,Gilberto Rego,23409,Francisco Olazar,Raoul Caudron,wc_1930,match_68cbf3f58928,team_2a52adc7b1da,team_e165d4f2174b,stadium_7d92fbf7c709,referee_7c00f883ccae,manager_8a04d6748b09,manager_4e437e3fee0c,home_win,0


,tournament_year,match_date,stage,home_team_name,away_team_name,home_team_norm,away_team_norm,home_score,away_score,stadium_name,city,referee_name,attendance,home_manager_name,away_manager_name,tournament_id,match_id,home_team_id,away_team_id,stadium_id,referee_id,home_manager_id,away_manager_id,result,target
959,2022,2022-12-10,Quarter-finals,England,France,england,france,1,2,"Al Bayt Stadium, Al Khor",Qatar,Wilton Sampaio,68895,Gareth Southgate,Didier Deschamps,wc_2022,match_eb04f9fd8d79,team_f48861ca24e2,team_e165d4f2174b,stadium_d50acc316ae6,referee_6e544b05845b,manager_af9fafa20e4d,manager_d20acac5d133,away_win,2
960,2022,2022-12-13,Semi-finals,Argentina,Croatia,argentina,croatia,3,0,"Lusail Iconic Stadium, Lusail",Qatar,Daniele Orsato,88966,Lionel Scaloni,Zlatko Dalić,wc_2022,match_f6887b5f5d6f,team_2a52adc7b1da,team_267640588a39,stadium_26ac9e09839d,referee_843dd13bf195,manager_0d01fd33aa3e,manager_3d6e4062a27d,home_win,0
961,2022,2022-12-14,Semi-finals,France,Morocco,france,morocco,2,0,"Al Bayt Stadium, Al Khor",Qatar,César Arturo Ramos,68294,Didier Deschamps,Hoalid Regragui,wc_2022,match_ebf4a0fe598d,team_e165d4f2174b,team_217b6e5f0a09,stadium_d50acc316ae6,referee_9bf52ecfb548,manager_d20acac5d133,manager_1e0aa3898b4b,home_win,0
962,2022,2022-12-17,Third-place match,Croatia,Morocco,croatia,morocco,2,1,"Khalifa International Stadium, Doha",Qatar,Abdulrahman Ibrahim Al Jassim,44137,Zlatko Dalić,Hoalid Regragui,wc_2022,match_c50654aea08c,team_267640588a39,team_217b6e5f0a09,stadium_e444b96557eb,referee_779de4b853a9,manager_3d6e4062a27d,manager_1e0aa3898b4b,home_win,0
963,2022,2022-12-18,Final,Argentina,France,argentina,france,3,3,"Lusail Iconic Stadium, Lusail",Qatar,Szymon Marciniak,88966,Lionel Scaloni,Didier Deschamps,wc_2022,match_438f9a929e58,team_2a52adc7b1da,team_e165d4f2174b,stadium_26ac9e09839d,referee_1fa3cb506236,manager_0d01fd33aa3e,manager_d20acac5d133,draw,1


[OK] Saved parquet: /Users/gaperov/Documents/University/SNA/data/processed/parquet/worldcup_matches_clean.parquet
[OK] Saved csv: /Users/gaperov/Documents/University/SNA/data/processed/csv/worldcup_matches_clean.csv


# Базовая проверка матчей

Что делает: 

Проверяет:
- количество матчей по турнирам;
- распределение target;
- пропуски по ключевым полям.

In [30]:
print("Matches by tournament year:")
display(matches["tournament_year"].value_counts().sort_index())

print("Target distribution:")
display(matches["result"].value_counts(dropna=False))

print("Missing values in key columns:")
key_cols = [
    "tournament_year",
    "match_date",
    "home_team_name",
    "away_team_name",
    "home_score",
    "away_score",
    "stadium_name",
    "referee_name",
]
display(matches[key_cols].isna().mean().sort_values(ascending=False))


Matches by tournament year:


tournament_year
1930    18
1934    17
1938    18
1950    22
1954    26
1958    35
1962    32
1966    32
1970    32
1974    38
1978    38
1982    52
1986    52
1990    52
1994    52
1998    64
2002    64
2006    64
2010    64
2014    64
2018    64
2022    64
Name: count, dtype: Int64

Target distribution:


result
home_win    532
away_win    218
draw        214
Name: count, dtype: int64

Missing values in key columns:


referee_name       0.264523
tournament_year    0.000000
match_date         0.000000
home_team_name     0.000000
away_team_name     0.000000
home_score         0.000000
away_score         0.000000
stadium_name       0.000000
dtype: float64

# Загрузка и обработка FIFA rankings

## Предпросмотр raw-файла FIFA rankings

Что делает: Показывает структуру файла fifa_mens_rank.csv.

In [31]:
rankings_preview = pd.read_csv(FIFA_RANK_PATH, nrows=5)

print("Raw FIFA ranking columns:")
print(list(rankings_preview.columns))

display(rankings_preview.head())


Raw FIFA ranking columns:
['date', 'semester', 'rank', 'team', 'acronym', 'total.points', 'previous.points', 'diff.points']


,date,semester,rank,team,acronym,total.points,previous.points,diff.points
0,2024,2,1,Argentina,ARG,1867.25,1883.50,-16.25
1,2024,2,2,France,FRA,1859.78,1859.85,-0.07
2,2024,2,3,Spain,ESP,1853.27,1844.33,8.94
3,2024,2,4,England,ENG,1813.81,1807.83,5.98
4,2024,2,5,Brazil,BRA,1775.85,1784.37,-8.52


# Функция загрузки FIFA rankings

Что делает: 
- Загружает FIFA rankings:
- стандартизирует колонки;
- создает ranking_date;
- нормализует названия команд;
- приводит числовые поля.

In [32]:
def load_fifa_rankings_full(path) -> pd.DataFrame:
    """
    Загружает FIFA rankings 1992-2024.
    
    Ожидаемая структура:
    date, semester, rank, team, acronym, total.points, previous.points, diff.points
    
    После standardize_columns:
    date, semester, rank, team, acronym, total_points, previous_points, diff_points
    """
    df = pd.read_csv(path)
    df = standardize_columns(df)

    print("[INFO] Available FIFA ranking columns:")
    print(list(df.columns))

    required = [
        "date",
        "semester",
        "rank",
        "team",
        "total_points",
        "previous_points",
        "diff_points",
    ]

    for c in required:
        if c not in df.columns:
            raise KeyError(
                f"Missing required ranking column: {c}. "
                f"Available: {list(df.columns)}"
            )

    out = df.copy()

    out["ranking_year"] = pd.to_numeric(out["date"], errors="coerce").astype("Int64")
    out["semester"] = pd.to_numeric(out["semester"], errors="coerce").astype("Int64")

    semester_date = out["semester"].map({1: "06-30", 2: "12-31"})

    out["ranking_date"] = pd.to_datetime(
        out["ranking_year"].astype(str) + "-" + semester_date.astype(str),
        errors="coerce",
    )

    out["team_norm"] = out["team"].apply(normalize_team_name)

    for c in ["rank", "total_points", "previous_points", "diff_points"]:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    keep = [
        "ranking_year",
        "semester",
        "ranking_date",
        "rank",
        "team",
        "team_norm",
        "acronym",
        "total_points",
        "previous_points",
        "diff_points",
    ]

    keep = [c for c in keep if c in out.columns]

    out = out[keep].drop_duplicates()
    out = out.sort_values(["ranking_date", "rank"]).reset_index(drop=True)

    return out


# Загрузка и сохранение очищенного FIFA rankings

Что делает: Загружает таблицу рейтингов, показывает диапазон дат и сохраняет processed-файл.

In [34]:
rankings = load_fifa_rankings_full(FIFA_RANK_PATH)

print(rankings.shape)
display(rankings.head())
display(rankings.tail())

print("Ranking date range:")
print(rankings["ranking_date"].min(), "->", rankings["ranking_date"].max())

save_table(rankings, PROCESSED_DIR_PARQUET / "fifa_rankings_clean.parquet")
save_table(rankings, PROCESSED_DIR_CSV / "fifa_rankings_clean.csv")


[INFO] Available FIFA ranking columns:
['date', 'semester', 'rank', 'team', 'acronym', 'total_points', 'previous_points', 'diff_points']
(13128, 10)


,ranking_year,semester,ranking_date,rank,team,team_norm,acronym,total_points,previous_points,diff_points
0,1992,2,1992-12-31,1,Germany,germany,GER,57.0,0.0,57.0
1,1992,2,1992-12-31,1,Italy,italy,ITA,57.0,0.0,57.0
2,1992,2,1992-12-31,3,Brazil,brazil,BRA,56.0,0.0,56.0
3,1992,2,1992-12-31,3,Sweden,sweden,SWE,56.0,0.0,56.0
4,1992,2,1992-12-31,5,England,england,ENG,55.0,0.0,55.0


,ranking_year,semester,ranking_date,rank,team,team_norm,acronym,total_points,previous_points,diff_points
13123,2024,2,2024-12-31,206,Turks and Caicos Islands,turks and caicos islands,TCA,803.98,803.98,0.00
13124,2024,2,2024-12-31,207,British Virgin Islands,british virgin islands,VGB,780.30,780.30,0.00
13125,2024,2,2024-12-31,208,US Virgin Islands,us virgin islands,VIR,779.71,779.71,0.00
13126,2024,2,2024-12-31,209,Anguilla,anguilla,AIA,769.31,769.31,0.00
13127,2024,2,2024-12-31,210,San Marino,san marino,SMR,747.42,737.04,10.38


Ranking date range:
1992-12-31 00:00:00 -> 2024-12-31 00:00:00
[OK] Saved parquet: /Users/gaperov/Documents/University/SNA/data/processed/parquet/fifa_rankings_clean.parquet
[OK] Saved csv: /Users/gaperov/Documents/University/SNA/data/processed/csv/fifa_rankings_clean.csv


# Проверка команд FIFA rankings

Что делает: Показывает уникальные команды рейтинга и помогает диагностировать проблемы с join по названиям.

In [35]:
print("Number of unique ranking teams:", rankings["team_norm"].nunique())

display(
    rankings[["team", "team_norm"]]
    .drop_duplicates()
    .sort_values("team_norm")
    .head(50)
)


Number of unique ranking teams: 226


,team,team_norm
4159,Afghanistan,afghanistan
85,Albania,albania
29,Algeria,algeria
2343,American Samoa,american samoa
1575,Andorra,andorra
100,Angola,angola
1766,Anguilla,anguilla
107,Antigua and Barbuda,antigua and barbuda
12392,Aotearoa New Zealand,aotearoa new zealand
9,Argentina,argentina


In [47]:
sum(rankings['team_norm'] == 'yugoslavia')

20

# Присоединение FIFA rankings к матчам

## Функция выбора pre-tournament snapshot

Что делает: Для каждого турнира выбирает последний доступный рейтинг до условной даты 1 июля года турнира.

Это защищает от leakage. Для ЧМ-2022 не будет использоваться 2022-S2, потому что snapshot после 1 июля не пройдет cutoff.

In [36]:
def get_pre_tournament_ranking_snapshot(rankings: pd.DataFrame, tournament_year: int) -> pd.DataFrame:
    """
    Выбирает ranking snapshot до турнира.

    MVP-правило:
    - берем последний ranking_date < 1 июля tournament_year;
    - для летних ЧМ это обычно year-S1;
    - для ЧМ-2022 это также защищает от 2022-S2.
    """
    cutoff = pd.Timestamp(f"{int(tournament_year)}-07-01")

    candidates = rankings[rankings["ranking_date"] < cutoff].copy()

    if candidates.empty:
        return pd.DataFrame(columns=list(rankings.columns) + ["snapshot_tournament_year"])

    max_date = candidates["ranking_date"].max()

    snapshot = candidates[candidates["ranking_date"] == max_date].copy()
    snapshot["snapshot_tournament_year"] = int(tournament_year)

    return snapshot


# Функция присоединения FIFA rankings к матчам

Что делает: 

Добавляет к каждому матчу рейтинги home и away команд:
- home_fifa_rank;
- away_fifa_rank;
- fifa_rank_diff;
- home_fifa_points;
- away_fifa_points;
- fifa_points_diff;
- fifa_momentum_diff;
- home_better_rank_flag.

In [37]:
def attach_fifa_rankings_by_tournament_snapshot(
    matches: pd.DataFrame,
    rankings: pd.DataFrame,
) -> pd.DataFrame:
    """
    Присоединяет FIFA rankings к матчам через pre-tournament snapshot.
    """
    out = matches.copy()

    years = sorted(out["tournament_year"].dropna().astype(int).unique())

    snapshots = []

    for y in years:
        snap = get_pre_tournament_ranking_snapshot(rankings, y)
        if len(snap) > 0:
            snapshots.append(snap)

    if snapshots:
        snapshot_df = pd.concat(snapshots, ignore_index=True)
    else:
        snapshot_df = pd.DataFrame()

    if snapshot_df.empty:
        print("[WARN] No ranking snapshots found.")
        return out

    home_rank = snapshot_df.rename(
        columns={
            "team_norm": "home_team_norm",
            "rank": "home_fifa_rank",
            "total_points": "home_fifa_points",
            "previous_points": "home_fifa_previous_points",
            "diff_points": "home_fifa_diff_points",
            "ranking_date": "home_fifa_ranking_date",
        }
    )

    away_rank = snapshot_df.rename(
        columns={
            "team_norm": "away_team_norm",
            "rank": "away_fifa_rank",
            "total_points": "away_fifa_points",
            "previous_points": "away_fifa_previous_points",
            "diff_points": "away_fifa_diff_points",
            "ranking_date": "away_fifa_ranking_date",
        }
    )

    home_cols = [
        "snapshot_tournament_year",
        "home_team_norm",
        "home_fifa_rank",
        "home_fifa_points",
        "home_fifa_previous_points",
        "home_fifa_diff_points",
        "home_fifa_ranking_date",
    ]

    away_cols = [
        "snapshot_tournament_year",
        "away_team_norm",
        "away_fifa_rank",
        "away_fifa_points",
        "away_fifa_previous_points",
        "away_fifa_diff_points",
        "away_fifa_ranking_date",
    ]

    out = out.merge(
        home_rank[home_cols],
        left_on=["tournament_year", "home_team_norm"],
        right_on=["snapshot_tournament_year", "home_team_norm"],
        how="left",
    )

    out = out.drop(columns=["snapshot_tournament_year"], errors="ignore")

    out = out.merge(
        away_rank[away_cols],
        left_on=["tournament_year", "away_team_norm"],
        right_on=["snapshot_tournament_year", "away_team_norm"],
        how="left",
    )

    out = out.drop(columns=["snapshot_tournament_year"], errors="ignore")

    out["fifa_rank_diff"] = out["away_fifa_rank"] - out["home_fifa_rank"]
    out["fifa_points_diff"] = out["home_fifa_points"] - out["away_fifa_points"]
    out["fifa_momentum_diff"] = out["home_fifa_diff_points"] - out["away_fifa_diff_points"]
    out["home_better_rank_flag"] = out["home_fifa_rank"] < out["away_fifa_rank"]

    return out


# Присоединение FIFA rankings

Что делает: Создает первую версию match_dataset с FIFA-признаками.

In [39]:
match_dataset = attach_fifa_rankings_by_tournament_snapshot(matches, rankings)

print(match_dataset.shape)
display(match_dataset.head())
display(match_dataset.tail())


(964, 39)


,tournament_year,match_date,stage,home_team_name,away_team_name,home_team_norm,away_team_norm,home_score,away_score,stadium_name,city,referee_name,attendance,home_manager_name,away_manager_name,tournament_id,match_id,home_team_id,away_team_id,stadium_id,referee_id,home_manager_id,away_manager_id,result,target,home_fifa_rank,home_fifa_points,home_fifa_previous_points,home_fifa_diff_points,home_fifa_ranking_date,away_fifa_rank,away_fifa_points,away_fifa_previous_points,away_fifa_diff_points,away_fifa_ranking_date,fifa_rank_diff,fifa_points_diff,fifa_momentum_diff,home_better_rank_flag
0,1930,1930-07-13,Group stage,France,Mexico,france,mexico,4,1,"Pocitos, Montevideo",Uruguay,Domingo Lombardi,4444,Raoul Caudron,Juan Luque,wc_1930,match_008af429d7d1,team_e165d4f2174b,team_4edfc924721a,stadium_910df19d9a18,referee_38c6c92bf039,manager_4e437e3fee0c,manager_bfd0d93fef9f,home_win,0,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,False
1,1930,1930-07-13,Group stage,United States,Belgium,united states,belgium,3,0,"Parque Central, Montevideo",Uruguay,Jose Macias,18346,Bob Millar,Hector Goetinck,wc_1930,match_1639b517874f,team_152649df347e,team_73fa01094ea6,stadium_7d92fbf7c709,referee_609b04cd8ef3,manager_1d1efef13b70,manager_be7e1c21c043,home_win,0,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,False
2,1930,1930-07-14,Group stage,Yugoslavia,Brazil,yugoslavia,brazil,2,1,"Parque Central, Montevideo",Uruguay,Anibal Tejada,24059,Bosko Simonovic,Pindaro De Carvalho,wc_1930,match_1f0a82595fdf,team_950ebd5e5210,team_6e5fa4d9c48c,stadium_7d92fbf7c709,referee_407514960fba,manager_50a77bcfc752,manager_1126c9d060a4,home_win,0,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,False
3,1930,1930-07-14,Group stage,Romania,Peru,romania,peru,3,1,"Pocitos, Montevideo",Uruguay,Alberto Warnken,2549,Octav Luchide,Francisco Bru,wc_1930,match_aff97a035956,team_89055f975d7d,team_32e8419a7ecb,stadium_910df19d9a18,referee_2e5451280212,manager_58ce448ba651,manager_1b6d188b83fe,home_win,0,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,False
4,1930,1930-07-15,Group stage,Argentina,France,argentina,france,1,0,"Parque Central, Montevideo",Uruguay,Gilberto Rego,23409,Francisco Olazar,Raoul Caudron,wc_1930,match_68cbf3f58928,team_2a52adc7b1da,team_e165d4f2174b,stadium_7d92fbf7c709,referee_7c00f883ccae,manager_8a04d6748b09,manager_4e437e3fee0c,home_win,0,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,False


,tournament_year,match_date,stage,home_team_name,away_team_name,home_team_norm,away_team_norm,home_score,away_score,stadium_name,city,referee_name,attendance,home_manager_name,away_manager_name,tournament_id,match_id,home_team_id,away_team_id,stadium_id,referee_id,home_manager_id,away_manager_id,result,target,home_fifa_rank,home_fifa_points,home_fifa_previous_points,home_fifa_diff_points,home_fifa_ranking_date,away_fifa_rank,away_fifa_points,away_fifa_previous_points,away_fifa_diff_points,away_fifa_ranking_date,fifa_rank_diff,fifa_points_diff,fifa_momentum_diff,home_better_rank_flag
959,2022,2022-12-10,Quarter-finals,England,France,england,france,1,2,"Al Bayt Stadium, Al Khor",Qatar,Wilton Sampaio,68895,Gareth Southgate,Didier Deschamps,wc_2022,match_eb04f9fd8d79,team_f48861ca24e2,team_e165d4f2174b,stadium_d50acc316ae6,referee_6e544b05845b,manager_af9fafa20e4d,manager_d20acac5d133,away_win,2,5.0,1737.46,1761.71,-24.25,2022-06-30,4.0,1764.85,1789.85,-25.00,2022-06-30,-1.0,-27.39,0.75,False
960,2022,2022-12-13,Semi-finals,Argentina,Croatia,argentina,croatia,3,0,"Lusail Iconic Stadium, Lusail",Qatar,Daniele Orsato,88966,Lionel Scaloni,Zlatko Dalić,wc_2022,match_f6887b5f5d6f,team_2a52adc7b1da,team_267640588a39,stadium_26ac9e09839d,referee_843dd13bf195,manager_0d01fd33aa3e,manager_3d6e4062a27d,home_win,0,3.0,1770.65,1765.13,5.52,2022-06-30,15.0,1632.15,1621.11,11.04,2022-06-30,12.0,138.50,-5.52,True
961,2022,2022-12-14,Semi-finals,France,Morocco,france,morocco,2,0,"Al Bayt Stadium, Al Khor",Qatar,César Arturo Ramos,68294,Didier Deschamps,Hoalid Regragui,wc_2022,match_ebf4a0fe598d,team_e165d4f2174b,team_217b6e5f0a09,stadium_d50acc316ae6,referee_9bf52ecfb548,manager_d20acac5d133,manager_1e0aa3898b4b,home_win,0,4.0,1764.85,1789.85,-25.00,2022-06-30,22.0,1558.90,1551.88,7.02,2022-06-30,18.0,205.95,-32.02,True
962,2022,2022-12-17,Third-place match,Croatia,Morocco,croatia,morocco,2,1,"Khalifa International Stadium, Doha",Qatar,Abdulrahman Ibrahim Al Jassim,44137,Zlatko Dalić,Hoalid Regragui,wc_2022,match_c50654aea08c,team_267640588a39,team_217b6e5f0a09,stadium_e444b96557eb,referee_779de4b853a9,manager_3d6e4062a27d,manager_1e0aa3898b4b,home_win,0,15.0,1632.15,1621.11,11.04,2022-06-30,22.0,1558.90,1551.88,7.02,2022-06-30,7.0,73.25,4.02,True
963,2022,2022-12-18,Final,Argentina,France,argentina,france,3,3,"Lusail Iconic Stadium, Lusail",Qatar,Szymon Marciniak,88966,Lionel Scaloni,Didier Deschamps,wc_2022,match_438f9a929e58,team_2a52adc7b1da,team_e165d4f2174b,stadium_26ac9e09839d,referee_1fa3cb506236,manager_0d01fd33aa3e,manager_d20acac5d133,draw,1,3.0,1770.65,1765.13,5.52,2022-06-30,4.0,1764.85,1789.85,-25.00,2022-06-30,1.0,5.80,30.52,True


# Проверка качества join FIFA rankings

Что делает: Считает долю матчей без рейтинга home/away по каждому турниру.

До 1994 рейтингов ожидаемо не будет или будет мало, поэтому основной modeling subset — 1994–2022.

In [41]:
def check_ranking_join_quality(df: pd.DataFrame) -> pd.DataFrame:
    """
    Проверяет качество присоединения FIFA rankings.
    """
    summary = (
        df.groupby("tournament_year")
        .agg(
            matches=("match_id", "count"),
            home_rank_missing=("home_fifa_rank", lambda s: s.isna().sum()),
            away_rank_missing=("away_fifa_rank", lambda s: s.isna().sum()),
        )
        .reset_index()
    )

    summary["home_rank_missing_rate"] = summary["home_rank_missing"] / summary["matches"]
    summary["away_rank_missing_rate"] = summary["away_rank_missing"] / summary["matches"]

    return summary


ranking_quality = check_ranking_join_quality(match_dataset)

display(ranking_quality)

save_table(ranking_quality, PROCESSED_DIR_CSV / "ranking_join_quality.csv")


,tournament_year,matches,home_rank_missing,away_rank_missing,home_rank_missing_rate,away_rank_missing_rate
0,1930,18,18,18,1.000000,1.000000
1,1934,17,17,17,1.000000,1.000000
2,1938,18,18,18,1.000000,1.000000
3,1950,22,22,22,1.000000,1.000000
4,1954,26,26,26,1.000000,1.000000
5,1958,35,35,35,1.000000,1.000000
6,1962,32,32,32,1.000000,1.000000
7,1966,32,32,32,1.000000,1.000000
8,1970,32,32,32,1.000000,1.000000
9,1974,38,38,38,1.000000,1.000000


[OK] Saved csv: /Users/gaperov/Documents/University/SNA/data/processed/csv/ranking_join_quality.csv


# Диагностика неприсоединившихся команд 1994–2022

Что делает: Показывает команды, по которым не нашелся FIFA ranking в modeling period. Это помогает расширять TEAM_ALIASES.

In [42]:
modeling_tmp = match_dataset[
    (match_dataset["tournament_year"] >= MODELING_START_YEAR)
    & (match_dataset["tournament_year"] <= MODELING_END_YEAR)
].copy()

missing_home = modeling_tmp.loc[
    modeling_tmp["home_fifa_rank"].isna(),
    ["tournament_year", "home_team_name", "home_team_norm"]
].rename(
    columns={
        "home_team_name": "team_name",
        "home_team_norm": "team_norm",
    }
)

missing_away = modeling_tmp.loc[
    modeling_tmp["away_fifa_rank"].isna(),
    ["tournament_year", "away_team_name", "away_team_norm"]
].rename(
    columns={
        "away_team_name": "team_name",
        "away_team_norm": "team_norm",
    }
)

missing_teams = pd.concat([missing_home, missing_away], ignore_index=True).drop_duplicates()

print("Missing ranking teams in modeling period:")
display(missing_teams.sort_values(["tournament_year", "team_norm"]))


Missing ranking teams in modeling period:


,tournament_year,team_name,team_norm
0,1998,FR Yugoslavia,fr yugoslavia
1,2002,Türkiye,türkiye


# Rolling features

## Функция подсчета очков

Что делает: Возвращает очки команды за матч:
- победа — 3;
- ничья — 1;
- поражение — 0.

In [48]:
def points_for(goals_for, goals_against):
    """
    Очки за матч с точки зрения одной команды.
    """
    if pd.isna(goals_for) or pd.isna(goals_against):
        return np.nan

    if goals_for > goals_against:
        return 3

    if goals_for == goals_against:
        return 1

    return 0


# Функция расчета rolling features

Что делает: Считает pre-match признаки строго на основе матчей, которые были до текущего матча:
- историческая форма на ЧМ;
- ast 3/5/10 матчей;
- накопленная форма внутри турнира;
- rest days.

In [49]:
def compute_rolling_features(matches: pd.DataFrame, windows=(3, 5, 10)) -> pd.DataFrame:
    """
    Считает rolling features без leakage.

    Для каждой команды признаки считаются до обновления истории текущим матчем.
    """
    df = matches.sort_values(
        ["match_date", "tournament_year", "match_id"],
        na_position="last"
    ).copy()

    team_history = defaultdict(list)
    tournament_history = defaultdict(list)
    last_match_date_in_tournament = {}

    rows = []

    for _, r in df.iterrows():
        row = r.to_dict()

        teams = [
            ("home", r["home_team_id"], r["home_score"], r["away_score"]),
            ("away", r["away_team_id"], r["away_score"], r["home_score"]),
        ]

        for side, team_id, gf, ga in teams:
            hist = team_history[team_id]

            row[f"{side}_wc_matches_before"] = len(hist)
            row[f"{side}_wc_points_before"] = sum(x["points"] for x in hist if pd.notna(x["points"]))
            row[f"{side}_wc_goals_for_before"] = sum(x["gf"] for x in hist if pd.notna(x["gf"]))
            row[f"{side}_wc_goals_against_before"] = sum(x["ga"] for x in hist if pd.notna(x["ga"]))

            row[f"{side}_wc_goal_diff_before"] = (
                row[f"{side}_wc_goals_for_before"] -
                row[f"{side}_wc_goals_against_before"]
            )

            wins = sum(1 for x in hist if x["points"] == 3)

            row[f"{side}_wc_wins_before"] = wins
            row[f"{side}_wc_win_rate_before"] = wins / len(hist) if len(hist) > 0 else np.nan

            for n in windows:
                lastn = hist[-n:]

                row[f"{side}_last{n}_matches"] = len(lastn)
                row[f"{side}_last{n}_points"] = sum(x["points"] for x in lastn if pd.notna(x["points"]))
                row[f"{side}_last{n}_goals_for"] = sum(x["gf"] for x in lastn if pd.notna(x["gf"]))
                row[f"{side}_last{n}_goals_against"] = sum(x["ga"] for x in lastn if pd.notna(x["ga"]))

                row[f"{side}_last{n}_goal_diff"] = (
                    row[f"{side}_last{n}_goals_for"] -
                    row[f"{side}_last{n}_goals_against"]
                )

                row[f"{side}_last{n}_win_rate"] = (
                    sum(1 for x in lastn if x["points"] == 3) / len(lastn)
                    if len(lastn) > 0
                    else np.nan
                )

            key = (team_id, r["tournament_id"])
            thist = tournament_history[key]

            row[f"{side}_tournament_matches_before"] = len(thist)
            row[f"{side}_tournament_points_before"] = sum(x["points"] for x in thist if pd.notna(x["points"]))
            row[f"{side}_tournament_goals_for_before"] = sum(x["gf"] for x in thist if pd.notna(x["gf"]))
            row[f"{side}_tournament_goals_against_before"] = sum(x["ga"] for x in thist if pd.notna(x["ga"]))

            row[f"{side}_tournament_goal_diff_before"] = (
                row[f"{side}_tournament_goals_for_before"] -
                row[f"{side}_tournament_goals_against_before"]
            )

            prev_date = last_match_date_in_tournament.get(key)

            if pd.notna(r["match_date"]) and prev_date is not None:
                row[f"{side}_rest_days"] = (r["match_date"] - prev_date).days
                row[f"{side}_first_match_in_tournament"] = False
            else:
                row[f"{side}_rest_days"] = np.nan
                row[f"{side}_first_match_in_tournament"] = True

        rows.append(row)

        # Обновляем историю только после извлечения признаков текущего матча
        home_points = points_for(r["home_score"], r["away_score"])
        away_points = points_for(r["away_score"], r["home_score"])

        team_history[r["home_team_id"]].append(
            {
                "match_id": r["match_id"],
                "date": r["match_date"],
                "gf": r["home_score"],
                "ga": r["away_score"],
                "points": home_points,
            }
        )

        team_history[r["away_team_id"]].append(
            {
                "match_id": r["match_id"],
                "date": r["match_date"],
                "gf": r["away_score"],
                "ga": r["home_score"],
                "points": away_points,
            }
        )

        home_key = (r["home_team_id"], r["tournament_id"])
        away_key = (r["away_team_id"], r["tournament_id"])

        tournament_history[home_key].append(
            {
                "match_id": r["match_id"],
                "date": r["match_date"],
                "gf": r["home_score"],
                "ga": r["away_score"],
                "points": home_points,
            }
        )

        tournament_history[away_key].append(
            {
                "match_id": r["match_id"],
                "date": r["match_date"],
                "gf": r["away_score"],
                "ga": r["home_score"],
                "points": away_points,
            }
        )

        if pd.notna(r["match_date"]):
            last_match_date_in_tournament[home_key] = r["match_date"]
            last_match_date_in_tournament[away_key] = r["match_date"]

    return pd.DataFrame(rows)


# Расчет rolling features

Что делает: Добавляет rolling-признаки к match_dataset.

In [51]:
match_dataset = compute_rolling_features(match_dataset)

print(match_dataset.shape)

rolling_cols = [
    "home_wc_matches_before",
    "away_wc_matches_before",
    "home_last5_points",
    "away_last5_points",
    "home_tournament_points_before",
    "away_tournament_points_before",
    "home_rest_days",
    "away_rest_days",
]

display(match_dataset[["tournament_year", "match_date", "home_team_name", "away_team_name"] + rolling_cols].tail(20))


(964, 103)


,tournament_year,match_date,home_team_name,away_team_name,home_wc_matches_before,away_wc_matches_before,home_last5_points,away_last5_points,home_tournament_points_before,away_tournament_points_before,home_rest_days,away_rest_days
944,2022,2022-12-02,Korea Republic,Portugal,36,32,4,10,1,6,4.0,4.0
945,2022,2022-12-02,Ghana,Uruguay,14,58,4,7,3,1,4.0,4.0
946,2022,2022-12-02,Serbia,Switzerland,8,39,4,7,1,3,4.0,4.0
947,2022,2022-12-02,Cameroon,Brazil,25,111,1,12,1,6,4.0,4.0
948,2022,2022-12-03,Argentina,Australia,84,19,9,7,6,6,3.0,3.0
949,2022,2022-12-03,Netherlands,United States,53,36,11,5,7,5,4.0,4.0
950,2022,2022-12-04,England,Senegal,72,11,7,7,7,6,5.0,5.0
951,2022,2022-12-04,France,Poland,69,37,12,7,6,4,4.0,4.0
952,2022,2022-12-05,Japan,Croatia,24,26,6,8,6,5,4.0,4.0
953,2022,2022-12-05,Brazil,Korea Republic,112,37,9,7,6,4,3.0,3.0


# Создание metadata-таблиц

## Создание таблицы teams

Что делает: Создает справочник команд из home/away команд.

In [52]:
def build_teams_table(matches: pd.DataFrame) -> pd.DataFrame:
    """
    Создает справочник команд.
    """
    home = matches[["home_team_id", "home_team_name", "home_team_norm"]].rename(
        columns={
            "home_team_id": "team_id",
            "home_team_name": "team_name",
            "home_team_norm": "team_name_normalized",
        }
    )

    away = matches[["away_team_id", "away_team_name", "away_team_norm"]].rename(
        columns={
            "away_team_id": "team_id",
            "away_team_name": "team_name",
            "away_team_norm": "team_name_normalized",
        }
    )

    teams = pd.concat([home, away], ignore_index=True)
    teams = teams.drop_duplicates("team_id")
    teams = teams.sort_values("team_name_normalized").reset_index(drop=True)

    return teams


teams = build_teams_table(match_dataset)

print(teams.shape)
display(teams.head())

save_table(teams, PROCESSED_DIR_PARQUET / "teams.parquet")
save_table(teams, PROCESSED_DIR_CSV / "teams.csv")


(86, 3)


,team_id,team_name,team_name_normalized
0,team_49f4fdc64da0,Algeria,algeria
1,team_e5e0bd43b7f1,Angola,angola
2,team_2a52adc7b1da,Argentina,argentina
3,team_fb2a54d63748,Australia,australia
4,team_6cd0c266688a,Austria,austria


[OK] Saved parquet: /Users/gaperov/Documents/University/SNA/data/processed/parquet/teams.parquet
[OK] Saved csv: /Users/gaperov/Documents/University/SNA/data/processed/csv/teams.csv


# Создание таблицы tournaments

Что делает: 

Создает справочник турниров:
- год;
- дата начала;
- дата окончания;
- количество матчей;
- количество команд.

In [53]:
def build_tournaments_table(matches: pd.DataFrame) -> pd.DataFrame:
    """
    Создает таблицу турниров.
    """
    rows = []

    for year, g in matches.groupby("tournament_year"):
        teams_set = set(g["home_team_id"]).union(set(g["away_team_id"]))

        rows.append(
            {
                "tournament_id": f"wc_{int(year)}",
                "tournament_year": int(year),
                "tournament_name": f"FIFA World Cup {int(year)}",
                "start_date": g["match_date"].min(),
                "end_date": g["match_date"].max(),
                "num_matches": g["match_id"].nunique(),
                "num_teams": len(teams_set),
            }
        )

    tournaments = pd.DataFrame(rows)
    tournaments = tournaments.sort_values("tournament_year").reset_index(drop=True)

    return tournaments


tournaments = build_tournaments_table(match_dataset)

print(tournaments.shape)
display(tournaments.head())

save_table(tournaments, PROCESSED_DIR_PARQUET / "tournaments.parquet")
save_table(tournaments, PROCESSED_DIR_CSV / "tournaments.csv")


(22, 7)


,tournament_id,tournament_year,tournament_name,start_date,end_date,num_matches,num_teams
0,wc_1930,1930,FIFA World Cup 1930,1930-07-13,1930-07-30,18,13
1,wc_1934,1934,FIFA World Cup 1934,1934-05-27,1934-06-10,17,16
2,wc_1938,1938,FIFA World Cup 1938,1938-06-04,1938-06-19,18,15
3,wc_1950,1950,FIFA World Cup 1950,1950-06-24,1950-07-16,22,13
4,wc_1954,1954,FIFA World Cup 1954,1954-06-16,1954-07-04,26,16


[OK] Saved parquet: /Users/gaperov/Documents/University/SNA/data/processed/parquet/tournaments.parquet
[OK] Saved csv: /Users/gaperov/Documents/University/SNA/data/processed/csv/tournaments.csv


# Создание таблицы team_tournament_features

Что делает: Создает таблицу “команда в турнире” с pre-tournament рейтингами.
Одна строка:
- Team + Tournament


In [55]:
def build_team_tournament_features(match_dataset: pd.DataFrame) -> pd.DataFrame:
    """
    Создает признаки команды в конкретном турнире.
    """
    rows = []

    for _, r in match_dataset.iterrows():
        rows.append(
            {
                "team_id": r["home_team_id"],
                "team_name": r["home_team_name"],
                "team_norm": r["home_team_norm"],
                "tournament_id": r["tournament_id"],
                "tournament_year": r["tournament_year"],
                "fifa_rank_pre_tournament": r.get("home_fifa_rank"),
                "fifa_points_pre_tournament": r.get("home_fifa_points"),
                "fifa_previous_points_pre_tournament": r.get("home_fifa_previous_points"),
                "fifa_diff_points_pre_tournament": r.get("home_fifa_diff_points"),
                "fifa_ranking_date": r.get("home_fifa_ranking_date"),
            }
        )

        rows.append(
            {
                "team_id": r["away_team_id"],
                "team_name": r["away_team_name"],
                "team_norm": r["away_team_norm"],
                "tournament_id": r["tournament_id"],
                "tournament_year": r["tournament_year"],
                "fifa_rank_pre_tournament": r.get("away_fifa_rank"),
                "fifa_points_pre_tournament": r.get("away_fifa_points"),
                "fifa_previous_points_pre_tournament": r.get("away_fifa_previous_points"),
                "fifa_diff_points_pre_tournament": r.get("away_fifa_diff_points"),
                "fifa_ranking_date": r.get("away_fifa_ranking_date"),
            }
        )

    tt = pd.DataFrame(rows)
    tt = tt.drop_duplicates(["team_id", "tournament_id"]).reset_index(drop=True)

    tt["team_tournament_id"] = tt.apply(
        lambda r: make_hash_id("team_tournament", r["team_id"], r["tournament_id"]),
        axis=1,
    )

    return tt


team_tournament_features = build_team_tournament_features(match_dataset)

print(team_tournament_features.shape)
display(team_tournament_features.tail())

save_table(team_tournament_features, PROCESSED_DIR_PARQUET / "team_tournament_features.parquet")
save_table(team_tournament_features, PROCESSED_DIR_CSV / "team_tournament_features.csv")


(489, 11)


,team_id,team_name,team_norm,tournament_id,tournament_year,fifa_rank_pre_tournament,fifa_points_pre_tournament,fifa_previous_points_pre_tournament,fifa_diff_points_pre_tournament,fifa_ranking_date,team_tournament_id
484,team_3c6cfbc2415a,Korea Republic,south korea,wc_2022,2022,28.0,1526.20,1519.54,6.66,2022-06-30,team_tournament_3840790eec64
485,team_ae0eea6eb6d6,Portugal,portugal,wc_2022,2022,9.0,1678.65,1674.78,3.87,2022-06-30,team_tournament_646c1bd4863a
486,team_0e6049dbb2ce,Ghana,ghana,wc_2022,2022,60.0,1389.68,1387.36,2.32,2022-06-30,team_tournament_9d9629b9f049
487,team_6e5fa4d9c48c,Brazil,brazil,wc_2022,2022,1.0,1837.56,1832.69,4.87,2022-06-30,team_tournament_b000942ef726
488,team_a3867434e7dc,Serbia,serbia,wc_2022,2022,25.0,1549.53,1547.53,2.00,2022-06-30,team_tournament_b06be90e22cd


[OK] Saved parquet: /Users/gaperov/Documents/University/SNA/data/processed/parquet/team_tournament_features.parquet
[OK] Saved csv: /Users/gaperov/Documents/University/SNA/data/processed/csv/team_tournament_features.csv


# Text layer MVP

## Создание пустого текстового слоя

Что делает: Создает MVP-таблицу match_texts, где для каждого матча указано, что текст пока отсутствует.
Позже эту таблицу можно обогатить через:
GDELT;
Guardian;
Wikipedia;
FIFA archives;
BBC/ESPN/Reuters.

In [58]:
def build_empty_match_texts(matches: pd.DataFrame) -> pd.DataFrame:
    """
    Создает MVP text layer.
    Пока текстов нет, но структура уже готова для hybrid graph-text модели.
    """
    out = matches[
        [
            "match_id",
            "tournament_year",
            "match_date",
            "home_team_id",
            "away_team_id",
        ]
    ].copy()

    out["text_available"] = 0
    out["text_count"] = 0
    out["combined_pre_match_text"] = ""
    out["sources"] = ""
    out["latest_text_time"] = pd.NaT
    out["min_hours_before_match"] = np.nan

    return out


match_texts = build_empty_match_texts(match_dataset)

print(match_texts.shape)
display(match_texts.head())

save_table(match_texts, PROCESSED_DIR_PARQUET / "match_texts.parquet")
save_table(match_texts, PROCESSED_DIR_CSV / "match_texts.csv")


(964, 11)


,match_id,tournament_year,match_date,home_team_id,away_team_id,text_available,text_count,combined_pre_match_text,sources,latest_text_time,min_hours_before_match
0,match_008af429d7d1,1930,1930-07-13,team_e165d4f2174b,team_4edfc924721a,0,0,,,NaT,NaN
1,match_1639b517874f,1930,1930-07-13,team_152649df347e,team_73fa01094ea6,0,0,,,NaT,NaN
2,match_1f0a82595fdf,1930,1930-07-14,team_950ebd5e5210,team_6e5fa4d9c48c,0,0,,,NaT,NaN
3,match_aff97a035956,1930,1930-07-14,team_89055f975d7d,team_32e8419a7ecb,0,0,,,NaT,NaN
4,match_68cbf3f58928,1930,1930-07-15,team_2a52adc7b1da,team_e165d4f2174b,0,0,,,NaT,NaN


[OK] Saved parquet: /Users/gaperov/Documents/University/SNA/data/processed/parquet/match_texts.parquet
[OK] Saved csv: /Users/gaperov/Documents/University/SNA/data/processed/csv/match_texts.csv


# Функция фильтрации pre-match текстов

Что делает: Функция на будущее: если соберем статьи, она отфильтрует только те, которые опубликованы до матча.

In [59]:
def filter_pre_match_texts(texts: pd.DataFrame, matches: pd.DataFrame) -> pd.DataFrame:
    """
    Фильтрует тексты так, чтобы не было leakage.

    Ожидаемые поля texts:
    - match_id
    - published_at
    - raw_text / clean_text
    """
    texts = texts.copy()
    texts["published_at"] = pd.to_datetime(texts["published_at"], errors="coerce")

    m = matches[["match_id", "match_date"]].copy()

    out = texts.merge(m, on="match_id", how="left")

    out["is_pre_match"] = out["published_at"] < out["match_date"]

    out["hours_before_match"] = (
        out["match_date"] - out["published_at"]
    ).dt.total_seconds() / 3600

    out["leakage_risk_flag"] = ~out["is_pre_match"]

    return out[out["is_pre_match"]].copy()


№ Присоединение text flags к match_dataset

Что делает: Добавляет в главную таблицу признаки доступности текста:
- text_available;
- text_count.

In [60]:
text_flags = match_texts[
    ["match_id", "text_available", "text_count"]
].drop_duplicates("match_id")

match_dataset = match_dataset.merge(text_flags, on="match_id", how="left")

match_dataset["text_available"] = match_dataset["text_available"].fillna(0).astype(int)
match_dataset["text_count"] = match_dataset["text_count"].fillna(0).astype(int)

display(match_dataset[["match_id", "text_available", "text_count"]].head())


,match_id,text_available,text_count
0,match_008af429d7d1,0,0
1,match_1639b517874f,0,0
2,match_1f0a82595fdf,0,0
3,match_aff97a035956,0,0
4,match_68cbf3f58928,0,0


In [62]:
display(match_dataset)

,tournament_year,match_date,stage,home_team_name,away_team_name,home_team_norm,away_team_norm,home_score,away_score,stadium_name,city,referee_name,attendance,home_manager_name,away_manager_name,tournament_id,match_id,home_team_id,away_team_id,stadium_id,referee_id,home_manager_id,away_manager_id,result,target,home_fifa_rank,home_fifa_points,home_fifa_previous_points,home_fifa_diff_points,home_fifa_ranking_date,away_fifa_rank,away_fifa_points,away_fifa_previous_points,away_fifa_diff_points,away_fifa_ranking_date,fifa_rank_diff,fifa_points_diff,fifa_momentum_diff,home_better_rank_flag,home_wc_matches_before,home_wc_points_before,home_wc_goals_for_before,home_wc_goals_against_before,home_wc_goal_diff_before,home_wc_wins_before,home_wc_win_rate_before,home_last3_matches,home_last3_points,home_last3_goals_for,home_last3_goals_against,home_last3_goal_diff,home_last3_win_rate,home_last5_matches,home_last5_points,home_last5_goals_for,home_last5_goals_against,home_last5_goal_diff,home_last5_win_rate,home_last10_matches,home_last10_points,home_last10_goals_for,home_last10_goals_against,home_last10_goal_diff,home_last10_win_rate,home_tournament_matches_before,home_tournament_points_before,home_tournament_goals_for_before,home_tournament_goals_against_before,home_tournament_goal_diff_before,home_rest_days,home_first_match_in_tournament,away_wc_matches_before,away_wc_points_before,away_wc_goals_for_before,away_wc_goals_against_before,away_wc_goal_diff_before,away_wc_wins_before,away_wc_win_rate_before,away_last3_matches,away_last3_points,away_last3_goals_for,away_last3_goals_against,away_last3_goal_diff,away_last3_win_rate,away_last5_matches,away_last5_points,away_last5_goals_for,away_last5_goals_against,away_last5_goal_diff,away_last5_win_rate,away_last10_matches,away_last10_points,away_last10_goals_for,away_last10_goals_against,away_last10_goal_diff,away_last10_win_rate,away_tournament_matches_before,away_tournament_points_before,away_tournament_goals_for_before,away_tournament_goals_against_before,away_tournament_goal_diff_before,away_rest_days,away_first_match_in_tournament,text_available,text_count
0,1930,1930-07-13,Group stage,France,Mexico,france,mexico,4,1,"Pocitos, Montevideo",Uruguay,Domingo Lombardi,4444,Raoul Caudron,Juan Luque,wc_1930,match_008af429d7d1,team_e165d4f2174b,team_4edfc924721a,stadium_910df19d9a18,referee_38c6c92bf039,manager_4e437e3fee0c,manager_bfd0d93fef9f,home_win,0,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,False,0,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0,0,0,0,0,NaN,True,0,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0,0,0,0,0,NaN,True,0,0
1,1930,1930-07-13,Group stage,United States,Belgium,united states,belgium,3,0,"Parque Central, Montevideo",Uruguay,Jose Macias,18346,Bob Millar,Hector Goetinck,wc_1930,match_1639b517874f,team_152649df347e,team_73fa01094ea6,stadium_7d92fbf7c709,referee_609b04cd8ef3,manager_1d1efef13b70,manager_be7e1c21c043,home_win,0,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,False,0,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0,0,0,0,0,NaN,True,0,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0,0,0,0,0,NaN,True,0,0
2,1930,1930-07-14,Group stage,Yugoslavia,Brazil,yugoslavia,brazil,2,1,"Parque Central, Montevideo",Uruguay,Anibal Tejada,24059,Bosko Simonovic,Pindaro De Carvalho,wc_1930,match_1f0a82595fdf,team_950ebd5e5210,team_6e5fa4d9c48c,stadium_7d92fbf7c709,referee_407514960fba,manager_50a77bcfc752,manager_1126c9d060a4,home_win,0,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,False,0,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0,0,0,0,0,NaN,True,0,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0,0,0,0,0,NaN,True,0,0
3,1930,1930-07-14,Group stage,Romania,Peru,romania,peru,3,1,"Pocitos, Montevideo",Uruguay,Alberto Warnken,2549,Octav Luchide,Francisco Bru,wc_1930,match_aff97a035956,team_89055f975d7d,team_32e8419a7ecb,stadium_910df19d9a18,referee_2e5451280212,manager_58ce448ba651,manager_1b6d188b83fe,ho

# Сохранение главного датасета

## Сохранение полного historical dataset 1930–2022

Что делает: Сохраняет главную таблицу match_dataset для всех доступных турниров.

In [66]:
save_table(match_dataset, PROCESSED_DIR_PARQUET / "match_dataset.parquet")
save_table(match_dataset, PROCESSED_DIR_CSV / "match_dataset.csv")

print(match_dataset.shape)
display(match_dataset.head())


[OK] Saved parquet: /Users/gaperov/Documents/University/SNA/data/processed/parquet/match_dataset.parquet
[OK] Saved csv: /Users/gaperov/Documents/University/SNA/data/processed/csv/match_dataset.csv
(964, 105)


,tournament_year,match_date,stage,home_team_name,away_team_name,home_team_norm,away_team_norm,home_score,away_score,stadium_name,city,referee_name,attendance,home_manager_name,away_manager_name,tournament_id,match_id,home_team_id,away_team_id,stadium_id,referee_id,home_manager_id,away_manager_id,result,target,home_fifa_rank,home_fifa_points,home_fifa_previous_points,home_fifa_diff_points,home_fifa_ranking_date,away_fifa_rank,away_fifa_points,away_fifa_previous_points,away_fifa_diff_points,away_fifa_ranking_date,fifa_rank_diff,fifa_points_diff,fifa_momentum_diff,home_better_rank_flag,home_wc_matches_before,home_wc_points_before,home_wc_goals_for_before,home_wc_goals_against_before,home_wc_goal_diff_before,home_wc_wins_before,home_wc_win_rate_before,home_last3_matches,home_last3_points,home_last3_goals_for,home_last3_goals_against,home_last3_goal_diff,home_last3_win_rate,home_last5_matches,home_last5_points,home_last5_goals_for,home_last5_goals_against,home_last5_goal_diff,home_last5_win_rate,home_last10_matches,home_last10_points,home_last10_goals_for,home_last10_goals_against,home_last10_goal_diff,home_last10_win_rate,home_tournament_matches_before,home_tournament_points_before,home_tournament_goals_for_before,home_tournament_goals_against_before,home_tournament_goal_diff_before,home_rest_days,home_first_match_in_tournament,away_wc_matches_before,away_wc_points_before,away_wc_goals_for_before,away_wc_goals_against_before,away_wc_goal_diff_before,away_wc_wins_before,away_wc_win_rate_before,away_last3_matches,away_last3_points,away_last3_goals_for,away_last3_goals_against,away_last3_goal_diff,away_last3_win_rate,away_last5_matches,away_last5_points,away_last5_goals_for,away_last5_goals_against,away_last5_goal_diff,away_last5_win_rate,away_last10_matches,away_last10_points,away_last10_goals_for,away_last10_goals_against,away_last10_goal_diff,away_last10_win_rate,away_tournament_matches_before,away_tournament_points_before,away_tournament_goals_for_before,away_tournament_goals_against_before,away_tournament_goal_diff_before,away_rest_days,away_first_match_in_tournament,text_available,text_count
0,1930,1930-07-13,Group stage,France,Mexico,france,mexico,4,1,"Pocitos, Montevideo",Uruguay,Domingo Lombardi,4444,Raoul Caudron,Juan Luque,wc_1930,match_008af429d7d1,team_e165d4f2174b,team_4edfc924721a,stadium_910df19d9a18,referee_38c6c92bf039,manager_4e437e3fee0c,manager_bfd0d93fef9f,home_win,0,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,False,0,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0,0,0,0,0,NaN,True,0,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0,0,0,0,0,NaN,True,0,0
1,1930,1930-07-13,Group stage,United States,Belgium,united states,belgium,3,0,"Parque Central, Montevideo",Uruguay,Jose Macias,18346,Bob Millar,Hector Goetinck,wc_1930,match_1639b517874f,team_152649df347e,team_73fa01094ea6,stadium_7d92fbf7c709,referee_609b04cd8ef3,manager_1d1efef13b70,manager_be7e1c21c043,home_win,0,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,False,0,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0,0,0,0,0,NaN,True,0,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0,0,0,0,0,NaN,True,0,0
2,1930,1930-07-14,Group stage,Yugoslavia,Brazil,yugoslavia,brazil,2,1,"Parque Central, Montevideo",Uruguay,Anibal Tejada,24059,Bosko Simonovic,Pindaro De Carvalho,wc_1930,match_1f0a82595fdf,team_950ebd5e5210,team_6e5fa4d9c48c,stadium_7d92fbf7c709,referee_407514960fba,manager_50a77bcfc752,manager_1126c9d060a4,home_win,0,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,False,0,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0,0,0,0,0,NaN,True,0,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0,0,0,0,0,NaN,0,0,0,0,0,NaN,True,0,0
3,1930,1930-07-14,Group stage,Romania,Peru,romania,peru,3,1,"Pocitos, Montevideo",Uruguay,Alberto Warnken,2549,Octav Luchide,Francisco Bru,wc_1930,match_aff97a035956,team_89055f975d7d,team_32e8419a7ecb,stadium_910df19d9a18,referee_2e5451280212,manager_58ce448ba651,manager_1b6d188b83fe,ho

# Создание modeling subset 1994–2022

Что делает: 
- Создает основной датасет для моделирования.

Почему 1994–2022:
- FIFA rankings начинаются с 1992;
- ЧМ-1994 — первый ЧМ после появления рейтинга;
- для более ранних турниров рейтингов нет.

In [68]:
modeling_dataset = match_dataset[
    (match_dataset["tournament_year"] >= MODELING_START_YEAR)
    & (match_dataset["tournament_year"] <= MODELING_END_YEAR)
].copy()

print(modeling_dataset.shape)
display(modeling_dataset["tournament_year"].value_counts().sort_index())

save_table(modeling_dataset, PROCESSED_DIR_PARQUET / "match_dataset_1994_2022.parquet")
save_table(modeling_dataset, PROCESSED_DIR_CSV / "match_dataset_1994_2022.csv")


(500, 105)


tournament_year
1994    52
1998    64
2002    64
2006    64
2010    64
2014    64
2018    64
2022    64
Name: count, dtype: int64

[OK] Saved parquet: /Users/gaperov/Documents/University/SNA/data/processed/parquet/match_dataset_1994_2022.parquet
[OK] Saved csv: /Users/gaperov/Documents/University/SNA/data/processed/csv/match_dataset_1994_2022.csv


# Импорты для text pipeline

In [63]:
import os
import time
import json
import math
import requests
import pandas as pd
import numpy as np

from pathlib import Path
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from datetime import timedelta


# Конфигурация text layer

## Пути для текстовых данных

Что делает: Создает папки для raw и processed текстов.

In [134]:
TEXT_RAW_DIR = RAW_DIR / "text"
TEXT_PROCESSED_DIR = PROCESSED_DIR / "text"

TEXT_RAW_DIR.mkdir(parents=True, exist_ok=True)
TEXT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

GDELT_RAW_PATH = TEXT_RAW_DIR / "gdelt_articles_raw.jsonl"
GUARDIAN_RAW_PATH = TEXT_RAW_DIR / "guardian_articles_raw.jsonl"
WIKIPEDIA_RAW_PATH = TEXT_RAW_DIR / "wikipedia_pages_raw.jsonl"

ARTICLES_CLEAN_PATH_PARQUET = TEXT_PROCESSED_DIR / "articles_clean.parquet"
ARTICLES_CLEAN_PATH_CSV = TEXT_PROCESSED_DIR / "articles_clean.csv"


MATCH_TEXTS_DETAILED_PATH = TEXT_PROCESSED_DIR / "match_texts_detailed.parquet"
MATCH_TEXTS_AGG_PATH_PARQUET = TEXT_PROCESSED_DIR / "match_texts.parquet"
MATCH_TEXTS_AGG_PATH_CSV = TEXT_PROCESSED_DIR / "match_texts.csv"

print(TEXT_RAW_DIR)
print(TEXT_PROCESSED_DIR)


/Users/gaperov/Documents/University/SNA/data/raw/text
/Users/gaperov/Documents/University/SNA/data/processed/text


# API-ключи

Что делает: Подготавливает переменные для API-ключей.
Для Guardian нужен ключ:

# Загрузка modeling dataset для текстового слоя

## Загрузка match_dataset_1994_2022

Что делает: Загружает основной датасет матчей 1994–2022.

In [69]:
modeling_path_parquet = PROCESSED_DIR_PARQUET / "match_dataset_1994_2022.parquet"
modeling_path_csv = PROCESSED_DIR_CSV / "match_dataset_1994_2022.csv"

if modeling_path_parquet.exists():
    modeling_dataset = pd.read_parquet(modeling_path_parquet)
elif modeling_path_csv.exists():
    modeling_dataset = pd.read_csv(modeling_path_csv)
else:
    raise FileNotFoundError("match_dataset_1994_2022 not found. Run previous pipeline first.")

modeling_dataset["match_date"] = pd.to_datetime(modeling_dataset["match_date"], errors="coerce")

print(modeling_dataset.shape)
display(modeling_dataset.head())


(500, 105)


,tournament_year,match_date,stage,home_team_name,away_team_name,home_team_norm,away_team_norm,home_score,away_score,stadium_name,city,referee_name,attendance,home_manager_name,away_manager_name,tournament_id,match_id,home_team_id,away_team_id,stadium_id,referee_id,home_manager_id,away_manager_id,result,target,home_fifa_rank,home_fifa_points,home_fifa_previous_points,home_fifa_diff_points,home_fifa_ranking_date,away_fifa_rank,away_fifa_points,away_fifa_previous_points,away_fifa_diff_points,away_fifa_ranking_date,fifa_rank_diff,fifa_points_diff,fifa_momentum_diff,home_better_rank_flag,home_wc_matches_before,home_wc_points_before,home_wc_goals_for_before,home_wc_goals_against_before,home_wc_goal_diff_before,home_wc_wins_before,home_wc_win_rate_before,home_last3_matches,home_last3_points,home_last3_goals_for,home_last3_goals_against,home_last3_goal_diff,home_last3_win_rate,home_last5_matches,home_last5_points,home_last5_goals_for,home_last5_goals_against,home_last5_goal_diff,home_last5_win_rate,home_last10_matches,home_last10_points,home_last10_goals_for,home_last10_goals_against,home_last10_goal_diff,home_last10_win_rate,home_tournament_matches_before,home_tournament_points_before,home_tournament_goals_for_before,home_tournament_goals_against_before,home_tournament_goal_diff_before,home_rest_days,home_first_match_in_tournament,away_wc_matches_before,away_wc_points_before,away_wc_goals_for_before,away_wc_goals_against_before,away_wc_goal_diff_before,away_wc_wins_before,away_wc_win_rate_before,away_last3_matches,away_last3_points,away_last3_goals_for,away_last3_goals_against,away_last3_goal_diff,away_last3_win_rate,away_last5_matches,away_last5_points,away_last5_goals_for,away_last5_goals_against,away_last5_goal_diff,away_last5_win_rate,away_last10_matches,away_last10_points,away_last10_goals_for,away_last10_goals_against,away_last10_goal_diff,away_last10_win_rate,away_tournament_matches_before,away_tournament_points_before,away_tournament_goals_for_before,away_tournament_goals_against_before,away_tournament_goal_diff_before,away_rest_days,away_first_match_in_tournament,text_available,text_count
0,1994,1994-06-17,Group stage,Germany,Bolivia,germany,bolivia,1,0,"Soldier Field, Chicago",United States,Arturo Brizio Carter,63117,Berti Vogts,Xavier Azkargorta Uriarte,wc_1994,match_24ab401684d2,team_28ef36b35ae8,team_76781be1c79d,stadium_21a36eff12d1,referee_082a13461fbe,manager_fd5ff9057421,manager_0610795098b1,home_win,0,3.0,61.0,60.0,1.0,1994-06-30,40.0,36.0,35.0,1.0,1994-06-30,37.0,25.0,0.0,True,12,25,39,27,12,8,0.666667,3,9,11,3,8,1.000000,5,12,21,13,8,0.8,10,19,32,24,8,0.600000,0,0,0,0,0,NaN,True,3,0,0,16,-16,0,0.000000,3,0,0,16,-16,0.0,3,0,0,16,-16,0.0,3,0,0,16,-16,0.0,0,0,0,0,0,NaN,True,0,0
1,1994,1994-06-17,Group stage,Spain,Korea Republic,spain,south korea,2,2,"Cotton Bowl, Dallas",United States,Peter Mikkelsen,56247,Javier Clemente,Ho Kon Kim,wc_1994,match_38ef6f79f047,team_1747d4b2dd4b,team_3c6cfbc2415a,stadium_eb34e2ed62d2,referee_a1c8baf68392,manager_7f77762d181f,manager_49069ba49784,draw,1,6.0,60.0,56.0,4.0,1994-06-30,35.0,39.0,38.0,1.0,1994-06-30,29.0,21.0,3.0,True,32,46,43,38,5,13,0.406250,3,6,6,4,2,0.666667,5,8,7,5,2,0.4,10,18,17,8,9,0.500000,0,0,0,0,0,NaN,True,8,1,5,29,-24,0,0.000000,3,0,1,6,-5,0.0,5,1,4,10,-6,0.0,8,1,5,29,-24,0.0,0,0,0,0,0,NaN,True,0,0
2,1994,1994-06-18,Group stage,Colombia,Romania,colombia,romania,1,3,"Rose Bowl, Los Angeles",United States,Jamal Al Sharif,91856,Francisco Maturana,Anghel Iordanescu,wc_1994,match_ebe85d2cc918,team_25446782e2cc,team_89055f975d7d,stadium_55473e526b54,referee_af626b23b33d,manager_7f318ef155c5,manager_17ab1b183cef,away_win,2,14.0,53.0,51.0,2.0,1994-06-30,7.0,58.0,55.0,3.0,1994-06-30,-7.0,-5.0,-1.0,False,7,5,9,15,-6,1,0.142857,3,1,2,4,-2,0.000000,5,4,4,9,-5,0.2,7,5,9,15,-6,0.142857,0,0,0,0,0,NaN,True,12,12,16,20,-4,3,0.250000,3,2,2,3,-1,0.0,5,5,6,6,0,0.2,10,9,13,15,-2,0.2,0,0,0,0,0,NaN,True,0,0
3,1994,1994-06-18,Group stage,Italy,Republic of Ireland,italy,ireland,0,1,"Giant

# Проверка годов

Что делает: Проверяет, какие годы турниров есть в modeling dataset.

In [70]:
display(modeling_dataset["tournament_year"].value_counts().sort_index())


tournament_year
1994    52
1998    64
2002    64
2006    64
2010    64
2014    64
2018    64
2022    64
Name: count, dtype: int64

# Общие функции для JSONL и текстовой очистки

## JSONL writer/reader

Что делает: Создает функции для сохранения и чтения сырых статей в формате .jsonl.

In [71]:
def append_jsonl(records, path):
    """
    Добавляет список словарей в JSONL-файл.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "a", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def read_jsonl(path):
    """
    Читает JSONL-файл в DataFrame.
    """
    path = Path(path)

    if not path.exists():
        return pd.DataFrame()

    rows = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception:
                pass

    return pd.DataFrame(rows)


# Очистка текста

Что делает: Удаляет HTML, лишние пробелы и приводит текст к более чистому виду.

In [72]:
def strip_html(raw):
    """
    Удаляет HTML-теги.
    """
    if pd.isna(raw):
        return ""

    raw = str(raw)

    soup = BeautifulSoup(raw, "html.parser")
    text = soup.get_text(separator=" ")

    return text


def clean_article_text(text):
    """
    Базовая очистка текста статьи.
    """
    if pd.isna(text):
        return ""

    text = strip_html(text)
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    text = text.strip()

    return text


def make_text_id(source, url=None, title=None, published_at=None):
    """
    Создает стабильный text_id.
    """
    return make_hash_id("text", source, url, title, published_at)


# Безопасный HTTP GET

Что делает: Делает HTTP-запрос с timeout и retry. Используется для Guardian.

In [73]:
def safe_get_json(url, params=None, headers=None, timeout=30, sleep=1.0, max_retries=3):
    """
    Безопасный GET-запрос, возвращает JSON или None.
    """
    for attempt in range(max_retries):
        try:
            r = requests.get(url, params=params, headers=headers, timeout=timeout)

            if r.status_code == 200:
                time.sleep(sleep)
                return r.json()

            print(f"[WARN] status={r.status_code}, url={r.url}")

            if r.status_code in [429, 500, 502, 503, 504]:
                time.sleep(sleep * (attempt + 2))
                continue

            return None

        except Exception as e:
            print(f"[WARN] Request failed: {e}")
            time.sleep(sleep * (attempt + 2))

    return None


# Генерация поисковых запросов по матчам

## Функция создания query для матча

Что делает: Создает поисковые строки для матчей.

Например:
- "Argentina France World Cup 2022 preview"
- "Argentina vs France World Cup 2022"
- "Argentina France team news World Cup 2022"


In [74]:
def build_match_queries(row):
    """
    Генерирует поисковые запросы для одного матча.
    """
    home = str(row["home_team_name"])
    away = str(row["away_team_name"])
    year = int(row["tournament_year"])

    queries = [
        f'"{home}" "{away}" "World Cup" {year} preview',
        f'"{home}" vs "{away}" "World Cup" {year}',
        f'"{home}" "{away}" "team news" "World Cup" {year}',
        f'"{home}" "{away}" "injury" "World Cup" {year}',
        f'"{home}" "{away}" "tactical preview" "World Cup" {year}',
    ]

    return queries


# Создание таблицы match-query

Что делает: Создает таблицу, где каждая строка — один запрос для одного матча.

Можно ограничивать сбор по годам, например начать с 2022, потом 2018 и т.д.

In [75]:
def build_match_query_table(matches, years=None):
    """
    Создает таблицу поисковых запросов для матчей.
    """
    df = matches.copy()

    if years is not None:
        df = df[df["tournament_year"].isin(years)].copy()

    rows = []

    for _, r in df.iterrows():
        queries = build_match_queries(r)

        for q in queries:
            rows.append(
                {
                    "match_id": r["match_id"],
                    "tournament_year": int(r["tournament_year"]),
                    "match_date": r["match_date"],
                    "home_team_name": r["home_team_name"],
                    "away_team_name": r["away_team_name"],
                    "query": q,
                }
            )

    return pd.DataFrame(rows)


# Рекомендуется начинать с 2022 для отладки
match_queries_2022 = build_match_query_table(modeling_dataset, years=[2022])

print(match_queries_2022.shape)
display(match_queries_2022.head(10))


(320, 6)


,match_id,tournament_year,match_date,home_team_name,away_team_name,query
0,match_eba59d5d01d0,2022,2022-11-20,Qatar,Ecuador,"""Qatar"" ""Ecuador"" ""World Cup"" 2022 preview"
1,match_eba59d5d01d0,2022,2022-11-20,Qatar,Ecuador,"""Qatar"" vs ""Ecuador"" ""World Cup"" 2022"
2,match_eba59d5d01d0,2022,2022-11-20,Qatar,Ecuador,"""Qatar"" ""Ecuador"" ""team news"" ""World Cup"" 2022"
3,match_eba59d5d01d0,2022,2022-11-20,Qatar,Ecuador,"""Qatar"" ""Ecuador"" ""injury"" ""World Cup"" 2022"
4,match_eba59d5d01d0,2022,2022-11-20,Qatar,Ecuador,"""Qatar"" ""Ecuador"" ""tactical preview"" ""World Cu..."
5,match_0a8f5c4a4766,2022,2022-11-21,United States,Wales,"""United States"" ""Wales"" ""World Cup"" 2022 preview"
6,match_0a8f5c4a4766,2022,2022-11-21,United States,Wales,"""United States"" vs ""Wales"" ""World Cup"" 2022"
7,match_0a8f5c4a4766,2022,2022-11-21,United States,Wales,"""United States"" ""Wales"" ""team news"" ""World Cup..."
8,match_0a8f5c4a4766,2022,2022-11-21,United States,Wales,"""United States"" ""Wales"" ""injury"" ""World Cup"" 2022"
9,match_0a8f5c4a4766,2022,2022-11-21,United States,Wales,"""United States"" ""Wales"" ""tactical preview"" ""Wo..."


# Функция запроса к Guardian

Что делает: Ищет статьи Guardian по query в заданном окне дат.

In [80]:
def guardian_search_articles(
    query,
    from_date,
    to_date,
    api_key,
    page_size=20,
    page=1,
):
    """
    Поиск статей Guardian.
    """
    if not api_key:
        return []

    base_url = "https://content.guardianapis.com/search"

    params = {
        "q": query,
        "from-date": pd.to_datetime(from_date).strftime("%Y-%m-%d"),
        "to-date": pd.to_datetime(to_date).strftime("%Y-%m-%d"),
        "section": "football",
        "show-fields": "headline,trailText,bodyText,byline,shortUrl",
        "page-size": page_size,
        "page": page,
        "api-key": api_key,
    }

    data = safe_get_json(base_url, params=params, sleep=1.0, max_retries=3)

    if data is None:
        return []

    response = data.get("response", {})
    results = response.get("results", [])

    out = []

    for a in results:
        fields = a.get("fields", {})

        raw_text = fields.get("bodyText") or fields.get("trailText") or fields.get("headline") or ""

        out.append(
            {
                "source": "guardian",
                "url": a.get("webUrl") or fields.get("shortUrl"),
                "title": a.get("webTitle") or fields.get("headline"),
                "published_at": a.get("webPublicationDate"),
                "section_name": a.get("sectionName"),
                "type": a.get("type"),
                "language": "en",
                "raw_text": raw_text,
                "snippet": fields.get("trailText"),
                "byline": fields.get("byline"),
                "query": query,
            }
        )

    return out


# Сбор Guardian для выбранных годов

Что делает: Запускает сбор Guardian-статей для матчей выбранных лет.

In [82]:
GUARDIAN_API_KEY = os.getenv("GUARDIAN_API_KEY", "")

# Если хотите временно задать вручную:
# GUARDIAN_API_KEY = "PASTE_YOUR_GUARDIAN_KEY_HERE"

if GUARDIAN_API_KEY:
    print("[OK] Guardian API key is set.")
else:
    print("[WARN] Guardian API key is not set. Guardian collection will be skipped unless you set it.")


[OK] Guardian API key is set.


In [83]:
def collect_guardian_for_matches(
    matches,
    years=(2022,),
    days_before=14,
    page_size=10,
    output_path=GUARDIAN_RAW_PATH,
    api_key=GUARDIAN_API_KEY,
):
    """
    Собирает Guardian articles для матчей выбранных лет.
    """
    if not api_key:
        print("[WARN] Guardian API key is empty. Skipping Guardian collection.")
        return

    query_table = build_match_query_table(matches, years=list(years))

    all_count = 0

    for _, qrow in tqdm(query_table.iterrows(), total=len(query_table)):
        match_date = pd.to_datetime(qrow["match_date"])

        if pd.isna(match_date):
            continue

        from_date = match_date - pd.Timedelta(days=days_before)
        to_date = match_date - pd.Timedelta(days=1)

        articles = guardian_search_articles(
            query=qrow["query"],
            from_date=from_date,
            to_date=to_date,
            api_key=api_key,
            page_size=page_size,
        )

        records = []

        for a in articles:
            rec = {
                **a,
                "match_id": qrow["match_id"],
                "tournament_year": qrow["tournament_year"],
                "match_date": str(qrow["match_date"]),
                "home_team_name": qrow["home_team_name"],
                "away_team_name": qrow["away_team_name"],
                "collector": "guardian_open_platform",
                "collected_at": pd.Timestamp.utcnow().isoformat(),
            }
            records.append(rec)

        if records:
            append_jsonl(records, output_path)
            all_count += len(records)

    print(f"[OK] Collected Guardian records: {all_count}")


# Запуск Guardian-сбора

Что делает: Собирает Guardian-статьи. Сначала рекомендуемый запуск — только 2022.

In [102]:
collect_guardian_for_matches(
    modeling_dataset,
    years=(1994,),
    days_before=14,
    page_size=5,
    output_path=GUARDIAN_RAW_PATH,
    api_key=GUARDIAN_API_KEY,
)


  0%|          | 0/260 [00:00<?, ?it/s]

/var/folders/w8/c9mrpvlj2v1g7fmy3l4lwy8r0000gn/T/ipykernel_47095/758989852.py:48: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  "collected_at": pd.Timestamp.utcnow().isoformat(),


[OK] Collected Guardian records: 50


# Проверка Guardian-данных

Что делает: Загружает Guardian JSONL и показывает coverage.

In [103]:
guardian_raw = read_jsonl(GUARDIAN_RAW_PATH)

print(guardian_raw.shape)
display(guardian_raw.tail())

if not guardian_raw.empty:
    display(guardian_raw["tournament_year"].value_counts().sort_index())
    display(guardian_raw["section_name"].value_counts(dropna=False).tail(20))


(10790, 18)


,source,url,title,published_at,section_name,type,language,raw_text,snippet,byline,query,match_id,tournament_year,match_date,home_team_name,away_team_name,collector,collected_at
10785,guardian,https://www.theguardian.com/football/1994/jul/...,Argentina fall to Hagi magic,1994-07-04T16:13:35Z,Football,article,en,"Diego Maradona, responsible for the greatest s...",<p><strong>1994 - Romania 3-2 Argentina:</stro...,Paul Wilson in Pasadena,"""Brazil"" ""Italy"" ""World Cup"" 1994 preview",match_01a014e4b89e,1994,1994-07-17 00:00:00,Brazil,Italy,guardian_open_platform,2026-05-23T05:17:05.717693+00:00
10786,guardian,https://www.theguardian.com/football/1994/jul/...,Argentina fall to Hagi magic,1994-07-04T16:13:35Z,Football,article,en,"Diego Maradona, responsible for the greatest s...",<p><strong>1994 - Romania 3-2 Argentina:</stro...,Paul Wilson in Pasadena,"""Brazil"" vs ""Italy"" ""World Cup"" 1994",match_01a014e4b89e,1994,1994-07-17 00:00:00,Brazil,Italy,guardian_open_platform,2026-05-23T05:17:06.986294+00:00
10787,guardian,https://www.theguardian.com/football/1994/jul/...,Argentina fall to Hagi magic,1994-07-04T16:13:35Z,Football,article,en,"Diego Maradona, responsible for the greatest s...",<p><strong>1994 - Romania 3-2 Argentina:</stro...,Paul Wilson in Pasadena,"""Brazil"" ""Italy"" ""team news"" ""World Cup"" 1994",match_01a014e4b89e,1994,1994-07-17 00:00:00,Brazil,Italy,guardian_open_platform,2026-05-23T05:17:08.278063+00:00
10788,guardian,https://www.theguardian.com/football/1994/jul/...,Argentina fall to Hagi magic,1994-07-04T16:13:35Z,Football,article,en,"Diego Maradona, responsible for the greatest s...",<p><strong>1994 - Romania 3-2 Argentina:</stro...,Paul Wilson in Pasadena,"""Brazil"" ""Italy"" ""injury"" ""World Cup"" 1994",match_01a014e4b89e,1994,1994-07-17 00:00:00,Brazil,Italy,guardian_open_platform,2026-05-23T05:17:09.562213+00:00
10789,guardian,https://www.theguardian.com/football/1994/jul/...,Argentina fall to Hagi magic,1994-07-04T16:13:35Z,Football,article,en,"Diego Maradona, responsible for the greatest s...",<p><strong>1994 - Romania 3-2 Argentina:</stro...,Paul Wilson in Pasadena,"""Brazil"" ""Italy"" ""tactical preview"" ""World Cup...",match_01a014e4b89e,1994,1994-07-17 00:00:00,Brazil,Italy,guardian_open_platform,2026-05-23T05:17:10.838594+00:00


tournament_year
1994      50
1998      40
2002    1600
2006    1100
2010    1600
2014    1600
2018    1600
2022    3200
Name: count, dtype: int64

section_name
Football    10790
Name: count, dtype: int64

# Объединение raw-статей из разных источников

## Загрузка всех raw text sources

Что делает: Загружает Guardian raw JSONL в отдельные DataFrame.

In [104]:
guardian_raw = read_jsonl(GUARDIAN_RAW_PATH)

print("Guardian:", guardian_raw.shape)



Guardian: (10790, 18)


# Унификация схемы статей

In [105]:
def standardize_article_schema(df, default_source=None):
    """
    Приводит статьи из разных источников к единой схеме.
    """
    if df is None or df.empty:
        return pd.DataFrame(
            columns=[
                "text_id",
                "match_id",
                "tournament_year",
                "match_date",
                "home_team_name",
                "away_team_name",
                "source",
                "url",
                "title",
                "published_at",
                "language",
                "query",
                "text_type",
                "raw_text",
                "snippet",
                "clean_text",
                "is_pre_match",
                "hours_before_match",
                "leakage_risk_flag",
            ]
        )

    out = df.copy()

    for c in [
        "match_id",
        "tournament_year",
        "match_date",
        "home_team_name",
        "away_team_name",
        "source",
        "url",
        "title",
        "published_at",
        "language",
        "query",
        "text_type",
        "raw_text",
        "snippet",
        "is_pre_match",
        "leakage_risk_flag",
    ]:
        if c not in out.columns:
            out[c] = None

    if default_source is not None:
        out["source"] = out["source"].fillna(default_source)

    out["match_date"] = pd.to_datetime(out["match_date"], errors="coerce")
    out["published_at"] = pd.to_datetime(out["published_at"], errors="coerce", utc=True)

    # match_date делаем timezone-aware для корректного сравнения с published_at
    out["match_date_utc"] = pd.to_datetime(out["match_date"], errors="coerce", utc=True)

    out["raw_text"] = out["raw_text"].fillna("")
    out["snippet"] = out["snippet"].fillna("")

    # Если raw_text короткий, добавляем snippet
    out["combined_raw"] = (
        out["title"].fillna("").astype(str)
        + ". "
        + out["raw_text"].fillna("").astype(str)
        + " "
        + out["snippet"].fillna("").astype(str)
    )

    out["clean_text"] = out["combined_raw"].apply(clean_article_text)

    out["text_id"] = out.apply(
        lambda r: make_text_id(
            r.get("source"),
            r.get("url"),
            r.get("title"),
            r.get("published_at"),
        ),
        axis=1,
    )

    return out


# Объединение источников в articles_clean

Что делает: Создает общую таблицу всех текстов.

In [109]:
# gdelt_std = standardize_article_schema(gdelt_raw, default_source="gdelt")
guardian_std = standardize_article_schema(guardian_raw, default_source="guardian")
# wikipedia_std = standardize_article_schema(wikipedia_raw, default_source="wikipedia")

articles_clean = pd.concat(
    [guardian_std],
    ignore_index=True,
)

articles_clean = articles_clean.drop_duplicates("text_id").reset_index(drop=True)

print(articles_clean.shape)
display(articles_clean.tail())

save_table(articles_clean, ARTICLES_CLEAN_PATH_PARQUET)
save_table(articles_clean, ARTICLES_CLEAN_PATH_CSV)


(994, 25)


,source,url,title,published_at,section_name,type,language,raw_text,snippet,byline,query,match_id,tournament_year,match_date,home_team_name,away_team_name,collector,collected_at,text_type,is_pre_match,leakage_risk_flag,match_date_utc,combined_raw,clean_text,text_id
989,guardian,https://www.theguardian.com/football/2002/jun/...,Is there a World Cup conspiracy?,2002-06-25 17:13:10+00:00,Football,article,en,Our mailbag is heaving. Talk of conspiracy han...,<p>Our mailbag is heaving. Talk of conspiracy ...,Barry Glendenning,"""Brazil"" vs ""Türkiye"" ""World Cup"" 2002",match_614c8e1c75b8,2002,2002-06-26,Brazil,Türkiye,guardian_open_platform,2026-05-23T04:11:58.860575+00:00,None,None,None,2002-06-26 00:00:00+00:00,Is there a World Cup conspiracy?. Our mailbag ...,Is there a World Cup conspiracy?. Our mailbag ...,text_d5a7c1816c8b
990,guardian,https://www.theguardian.com/football/2002/jun/...,Ronaldo sets up Brazil-Germany final,2002-06-26 13:34:00+00:00,Football,article,en,A moment of magic from Ronaldo secured Brazil'...,<p>A moment of magic from Ronaldo secured Braz...,Staff and agencies,"""Germany"" ""Brazil"" ""World Cup"" 2002 preview",match_932b71124d7d,2002,2002-06-30,Germany,Brazil,guardian_open_platform,2026-05-23T04:12:11.138856+00:00,None,None,None,2002-06-30 00:00:00+00:00,Ronaldo sets up Brazil-Germany final. A moment...,Ronaldo sets up Brazil-Germany final. A moment...,text_70eb841236b7
991,guardian,https://www.theguardian.com/football/2002/jun/...,Italy's Collina to referee World Cup final,2002-06-27 10:30:12+00:00,Football,article,en,Italian referee Pierluigi Collina will take ch...,<p>Italian Uncle Fester lookalike Pierluigi Co...,Staff and agencies,"""Germany"" ""Brazil"" ""World Cup"" 2002 preview",match_932b71124d7d,2002,2002-06-30,Germany,Brazil,guardian_open_platform,2026-05-23T04:12:11.139103+00:00,None,None,None,2002-06-30 00:00:00+00:00,Italy's Collina to referee World Cup final. It...,Italy's Collina to referee World Cup final. It...,text_50100b581f99
992,guardian,https://www.theguardian.com/football/1998/jul/...,Penalties sink England again,1998-07-01 16:13:38+00:00,Football,article,en,"England are out of the World Cup, for a second...",<p><strong>1998 - Argentina 2-2 England (Argen...,David Lacey in St Etienne,"""Brazil"" ""Denmark"" ""World Cup"" 1998 preview",match_920ea64b49ef,1998,1998-07-03,Brazil,Denmark,guardian_open_platform,2026-05-23T05:10:26.334237+00:00,None,None,None,1998-07-03 00:00:00+00:00,Penalties sink England again. England are out ...,Penalties sink England again. England are out ...,text_64de325bcbda
993,guardian,https://www.theguardian.com/football/1994/jul/...,Argentina fall to Hagi magic,1994-07-04 16:13:35+00:00,Football,article,en,"Diego Maradona, responsible for the greatest s...",<p><strong>1994 - Romania 3-2 Argentina:</stro...,Paul Wilson in Pasadena,"""Nigeria"" ""Italy"" ""World Cup"" 1994 preview",match_42c217b3cc23,1994,1994-07-05,Nigeria,Italy,guardian_open_platform,2026-05-23T05:16:08.191082+00:00,None,None,None,1994-07-05 00:00:00+00:00,"Argentina fall to Hagi magic. Diego Maradona, ...","Argentina fall to Hagi magic. Diego Maradona, ...",text_57564f519a91


[OK] Saved parquet: /Users/gaperov/Documents/University/SNA/data/processed/text/articles_clean.parquet
[OK] Saved csv: /Users/gaperov/Documents/University/SNA/data/processed/text/articles_clean.csv


# Leakage-safe фильтрация pre-match статей

## Функция фильтрации pre-match статей

Что делает: Оставляет для моделирования только статьи:

published_at < match_date


In [111]:
def create_pre_match_articles_for_modeling(articles, matches):
    """
    Создает leakage-safe таблицу pre-match articles.

    Правила:
    - article must have match_id;
    - published_at must be known;
    - published_at < match_date;
    """
    articles = articles.copy()

    m = matches[["match_id", "match_date"]].copy()
    m["match_date"] = pd.to_datetime(m["match_date"], errors="coerce", utc=True)

    articles = articles.drop(columns=["match_date_utc"], errors="ignore")

    out = articles.merge(
        m.rename(columns={"match_date": "official_match_date_utc"}),
        on="match_id",
        how="left",
    )

    out["published_at"] = pd.to_datetime(out["published_at"], errors="coerce", utc=True)

    out["is_pre_match"] = out["published_at"] < out["official_match_date_utc"]

    out["hours_before_match"] = (
        out["official_match_date_utc"] - out["published_at"]
    ).dt.total_seconds() / 3600

    out["leakage_risk_flag"] = False

    # Нет match_id -> нельзя использовать для match-level predictive text
    out.loc[out["match_id"].isna(), "leakage_risk_flag"] = True

    # Нет published_at -> нельзя доказать pre-match
    out.loc[out["published_at"].isna(), "leakage_risk_flag"] = True

    # Не pre-match -> leakage
    out.loc[out["is_pre_match"] != True, "leakage_risk_flag"] = True

    predictive = out[
        (out["leakage_risk_flag"] == False)
        & (out["is_pre_match"] == True)
        & (out["clean_text"].str.len() > 30)
    ].copy()

    predictive = predictive.sort_values(
        ["match_id", "published_at"],
        ascending=[True, False]
    ).reset_index(drop=True)

    return predictive, out


# Применение leakage-safe фильтра

Что делает: Создает:
- pre_match_articles — безопасные тексты для моделирования;
- articles_with_flags — все тексты с флагами leakage.

In [112]:
pre_match_articles, articles_with_flags = create_pre_match_articles_for_modeling(
    articles_clean,
    modeling_dataset,
)

print("All articles with flags:", articles_with_flags.shape)
print("Predictive pre-match articles:", pre_match_articles.shape)

display(pre_match_articles.head())

save_table(articles_with_flags, TEXT_PROCESSED_DIR / "articles_with_flags.parquet")
save_table(pre_match_articles, TEXT_PROCESSED_DIR / "pre_match_articles.parquet")

save_table(articles_with_flags, TEXT_PROCESSED_DIR / "articles_with_flags.csv")
save_table(pre_match_articles, TEXT_PROCESSED_DIR / "pre_match_articles.csv")


All articles with flags: (994, 26)
Predictive pre-match articles: (994, 26)


,source,url,title,published_at,section_name,type,language,raw_text,snippet,byline,query,match_id,tournament_year,match_date,home_team_name,away_team_name,collector,collected_at,text_type,is_pre_match,leakage_risk_flag,combined_raw,clean_text,text_id,official_match_date_utc,hours_before_match
0,guardian,https://www.theguardian.com/football/2014/jun/...,World Cup 2014: Algeria’s Vahid Halilhodzic pr...,2014-06-25 13:42:00+00:00,Football,article,en,Vahid Halilhodzic had just seen Algeria produc...,After delivering their first finals win since ...,Nick Ames,"""Algeria"" ""Russia"" ""World Cup"" 2014 preview",match_02de699b385f,2014,2014-06-26,Algeria,Russia,guardian_open_platform,2026-05-22T21:31:52.140172+00:00,None,True,False,World Cup 2014: Algeria’s Vahid Halilhodzic pr...,World Cup 2014: Algeria’s Vahid Halilhodzic pr...,text_f413e65034bb,2014-06-26 00:00:00+00:00,10.300000
1,guardian,https://www.theguardian.com/football/2014/jun/...,South Korea v Algeria: World Cup 2014 - as it ...,2014-06-22 21:01:34+00:00,Football,liveblog,en,Who expected a classy attacking display from A...,<b>As it happened:</b> South Korea and Algeria...,Toby Moses,"""Algeria"" ""Russia"" ""World Cup"" 2014 preview",match_02de699b385f,2014,2014-06-26,Algeria,Russia,guardian_open_platform,2026-05-22T21:31:52.140141+00:00,None,True,False,South Korea v Algeria: World Cup 2014 - as it ...,South Korea v Algeria: World Cup 2014 - as it ...,text_04074cd9e5a6,2014-06-26 00:00:00+00:00,74.973889
2,guardian,https://www.theguardian.com/football/2014/jun/...,Belgium v Russia - World Cup 2014 as it happened,2014-06-22 17:53:34+00:00,Football,liveblog,en,"Well, that was utterly dire for about 85 minut...",<b>Minute-by-minute report:</b> Belgium are th...,Nick Miller,"""Algeria"" vs ""Russia"" ""World Cup"" 2014",match_02de699b385f,2014,2014-06-26,Algeria,Russia,guardian_open_platform,2026-05-22T21:31:53.505914+00:00,None,True,False,Belgium v Russia - World Cup 2014 as it happen...,Belgium v Russia - World Cup 2014 as it happen...,text_45695f95ae24,2014-06-26 00:00:00+00:00,78.107222
3,guardian,https://www.theguardian.com/football/video/201...,World Cup Show 2014: day 11 previews – video,2014-06-22 12:00:00+00:00,Football,video,en,<p>Nat Coombs is joined by Guardian journalist...,<p>Nat Coombs is joined by Guardian journalist...,NaN,"""Algeria"" ""Russia"" ""World Cup"" 2014 preview",match_02de699b385f,2014,2014-06-26,Algeria,Russia,guardian_open_platform,2026-05-22T21:31:52.140061+00:00,None,True,False,World Cup Show 2014: day 11 previews – video. ...,World Cup Show 2014: day 11 previews – video. ...,text_9b7aefd504b6,2014-06-26 00:00:00+00:00,84.000000
4,guardian,https://www.theguardian.com/football/live/2022...,Croatia 4-1 Canada: World Cup 2022 – as it hap...,2022-11-27 18:31:27+00:00,Football,liveblog,en,Bryan Graham was at the match today and here’s...,The 2018 finalists came back from an early def...,Beau Dure,"""Korea Republic"" vs ""Ghana"" ""World Cup"" 2022",match_03a870591a86,2022,2022-11-28,Korea Republic,Ghana,guardian_open_platform,2026-05-22T21:08:50.833521+00:00,None,True,False,Croatia 4-1 Canada: World Cup 2022 – as it hap...,Croatia 4-1 Canada: World Cup 2022 – as it hap...,text_3da7fbb86b9e,2022-11-28 00:00:00+00:00,5.475833


[OK] Saved parquet: /Users/gaperov/Documents/University/SNA/data/processed/text/articles_with_flags.parquet
[OK] Saved parquet: /Users/gaperov/Documents/University/SNA/data/processed/text/pre_match_articles.parquet
[OK] Saved csv: /Users/gaperov/Documents/University/SNA/data/processed/text/articles_with_flags.csv
[OK] Saved csv: /Users/gaperov/Documents/University/SNA/data/processed/text/pre_match_articles.csv


# Coverage pre-match статей

Что делает: Показывает, сколько матчей получили хотя бы один pre-match текст.

In [113]:
coverage = (
    pre_match_articles.groupby("tournament_year")
    .agg(
        articles=("text_id", "count"),
        matches_with_text=("match_id", "nunique"),
    )
    .reset_index()
)

matches_by_year = (
    modeling_dataset.groupby("tournament_year")
    .agg(total_matches=("match_id", "nunique"))
    .reset_index()
)

coverage = coverage.merge(matches_by_year, on="tournament_year", how="right")
coverage["articles"] = coverage["articles"].fillna(0).astype(int)
coverage["matches_with_text"] = coverage["matches_with_text"].fillna(0).astype(int)
coverage["match_text_coverage"] = coverage["matches_with_text"] / coverage["total_matches"]

display(coverage.sort_values("tournament_year"))


,tournament_year,articles,matches_with_text,total_matches,match_text_coverage
0,1994,1,1,52,0.019231
1,1998,1,1,64,0.015625
2,2002,132,59,64,0.921875
3,2006,112,39,64,0.609375
4,2010,228,61,64,0.953125
5,2014,229,62,64,0.968750
6,2018,124,54,64,0.843750
7,2022,167,52,64,0.812500


# Агрегация текстов до уровня матча

## Функция агрегации pre-match текстов

Что делает: Для каждого матча объединяет несколько статей в один текст:

In [114]:
def aggregate_pre_match_texts_to_match_level(
    pre_match_articles,
    matches,
    max_articles_per_match=5,
    max_chars_per_article=3000,
):
    """
    Агрегирует статьи до уровня матча.

    Берем последние max_articles_per_match статей перед матчем.
    """
    matches_base = matches[
        [
            "match_id",
            "tournament_year",
            "match_date",
            "home_team_id",
            "away_team_id",
            "home_team_name",
            "away_team_name",
        ]
    ].copy()

    if pre_match_articles.empty:
        out = matches_base.copy()
        out["text_available"] = 0
        out["text_count"] = 0
        out["combined_pre_match_text"] = ""
        out["sources"] = ""
        out["latest_text_time"] = pd.NaT
        out["min_hours_before_match"] = np.nan
        return out

    articles = pre_match_articles.copy()
    articles["published_at"] = pd.to_datetime(articles["published_at"], errors="coerce", utc=True)

    articles = articles.sort_values(
        ["match_id", "published_at"],
        ascending=[True, False]
    )

    articles["rank_within_match"] = articles.groupby("match_id").cumcount() + 1

    articles = articles[articles["rank_within_match"] <= max_articles_per_match].copy()

    articles["clean_text_short"] = articles["clean_text"].fillna("").str.slice(0, max_chars_per_article)

    grouped = (
        articles.groupby("match_id")
        .agg(
            text_count=("text_id", "count"),
            sources=("source", lambda s: ",".join(sorted(set([str(x) for x in s if pd.notna(x)])))),
            latest_text_time=("published_at", "max"),
            min_hours_before_match=("hours_before_match", "min"),
            combined_pre_match_text=(
                "clean_text_short",
                lambda texts: "\n\n[ARTICLE_SEP]\n\n".join([t for t in texts if isinstance(t, str) and len(t) > 0])
            ),
        )
        .reset_index()
    )

    out = matches_base.merge(grouped, on="match_id", how="left")

    out["text_count"] = out["text_count"].fillna(0).astype(int)
    out["text_available"] = (out["text_count"] > 0).astype(int)
    out["sources"] = out["sources"].fillna("")
    out["combined_pre_match_text"] = out["combined_pre_match_text"].fillna("")

    return out


# Создание match_texts

Что делает: Создает итоговую таблицу match_texts на уровне матча.

In [135]:
match_texts = aggregate_pre_match_texts_to_match_level(
    pre_match_articles,
    modeling_dataset,
    max_articles_per_match=5,
    max_chars_per_article=3000,
)

print(match_texts.shape)
display(match_texts.tail())

save_table(match_texts, MATCH_TEXTS_AGG_PATH_PARQUET)
save_table(match_texts, MATCH_TEXTS_AGG_PATH_CSV)


(500, 13)


,match_id,tournament_year,match_date,home_team_id,away_team_id,home_team_name,away_team_name,text_count,sources,latest_text_time,min_hours_before_match,combined_pre_match_text,text_available
495,match_eb04f9fd8d79,2022,2022-12-10,team_f48861ca24e2,team_e165d4f2174b,England,France,5,guardian,2022-12-08 16:32:10+00:00,31.463889,World Cup 2022: England and France players fac...,1
496,match_f6887b5f5d6f,2022,2022-12-13,team_2a52adc7b1da,team_267640588a39,Argentina,Croatia,5,guardian,2022-12-12 17:25:04+00:00,6.582222,World Cup 2022: Scaloni defends Argentina beha...,1
497,match_ebf4a0fe598d,2022,2022-12-14,team_e165d4f2174b,team_217b6e5f0a09,France,Morocco,5,guardian,2022-12-13 21:12:06+00:00,2.798333,Argentina 3-0 Croatia: World Cup 2022 semi-fin...,1
498,match_c50654aea08c,2022,2022-12-17,team_267640588a39,team_217b6e5f0a09,Croatia,Morocco,4,guardian,2022-12-16 00:00:36+00:00,23.990000,World Cup 2022 briefing: Argentina v France wi...,1
499,match_438f9a929e58,2022,2022-12-18,team_2a52adc7b1da,team_e165d4f2174b,Argentina,France,2,guardian,2022-12-15 17:07:17+00:00,54.878611,World Cup 2022: Argentina and France train the...,1


[OK] Saved parquet: /Users/gaperov/Documents/University/SNA/data/processed/text/match_texts.parquet
[OK] Saved csv: /Users/gaperov/Documents/University/SNA/data/processed/text/match_texts.csv


# Проверка coverage в match_texts

Что делает: Показывает долю матчей с текстами по годам.

In [136]:
match_text_coverage = (
    match_texts.groupby("tournament_year")
    .agg(
        matches=("match_id", "count"),
        matches_with_text=("text_available", "sum"),
        avg_text_count=("text_count", "mean"),
    )
    .reset_index()
)

match_text_coverage["coverage_rate"] = (
    match_text_coverage["matches_with_text"] / match_text_coverage["matches"]
)

display(match_text_coverage.sort_values("tournament_year"))


,tournament_year,matches,matches_with_text,avg_text_count,coverage_rate
0,1994,52,1,0.019231,0.019231
1,1998,64,1,0.015625,0.015625
2,2002,64,59,2.015625,0.921875
3,2006,64,39,1.718750,0.609375
4,2010,64,61,3.312500,0.953125
5,2014,64,62,3.453125,0.968750
6,2018,64,54,1.921875,0.843750
7,2022,64,52,2.421875,0.812500


# Внедрение текстов в match_dataset

## Удаление старых text columns

Что делает: Если в match_dataset уже были placeholder-колонки:
- text_available
- text_count

удаляет их, чтобы заменить настоящими.


In [137]:
text_cols_to_drop = [
    "text_available",
    "text_count",
    "combined_pre_match_text",
    "sources",
    "latest_text_time",
    "min_hours_before_match",
]

modeling_dataset_text = modeling_dataset.drop(columns=text_cols_to_drop, errors="ignore").copy()


# Merge match_texts в modeling dataset

Что делает: Добавляет текстовые поля к modeling_dataset.

In [138]:
text_merge_cols = [
    "match_id",
    "text_available",
    "text_count",
    "combined_pre_match_text",
    "sources",
    "latest_text_time",
    "min_hours_before_match",
]

modeling_dataset_text = modeling_dataset_text.merge(
    match_texts[text_merge_cols],
    on="match_id",
    how="left",
)

modeling_dataset_text["text_available"] = modeling_dataset_text["text_available"].fillna(0).astype(int)
modeling_dataset_text["text_count"] = modeling_dataset_text["text_count"].fillna(0).astype(int)
modeling_dataset_text["combined_pre_match_text"] = modeling_dataset_text["combined_pre_match_text"].fillna("")
modeling_dataset_text["sources"] = modeling_dataset_text["sources"].fillna("")


In [139]:
print(modeling_dataset_text.shape)
display(modeling_dataset_text.tail(20))

(500, 109)


,tournament_year,match_date,stage,home_team_name,away_team_name,home_team_norm,away_team_norm,home_score,away_score,stadium_name,city,referee_name,attendance,home_manager_name,away_manager_name,tournament_id,match_id,home_team_id,away_team_id,stadium_id,referee_id,home_manager_id,away_manager_id,result,target,home_fifa_rank,home_fifa_points,home_fifa_previous_points,home_fifa_diff_points,home_fifa_ranking_date,away_fifa_rank,away_fifa_points,away_fifa_previous_points,away_fifa_diff_points,away_fifa_ranking_date,fifa_rank_diff,fifa_points_diff,fifa_momentum_diff,home_better_rank_flag,home_wc_matches_before,home_wc_points_before,home_wc_goals_for_before,home_wc_goals_against_before,home_wc_goal_diff_before,home_wc_wins_before,home_wc_win_rate_before,home_last3_matches,home_last3_points,home_last3_goals_for,home_last3_goals_against,home_last3_goal_diff,home_last3_win_rate,home_last5_matches,home_last5_points,home_last5_goals_for,home_last5_goals_against,home_last5_goal_diff,home_last5_win_rate,home_last10_matches,home_last10_points,home_last10_goals_for,home_last10_goals_against,home_last10_goal_diff,home_last10_win_rate,home_tournament_matches_before,home_tournament_points_before,home_tournament_goals_for_before,home_tournament_goals_against_before,home_tournament_goal_diff_before,home_rest_days,home_first_match_in_tournament,away_wc_matches_before,away_wc_points_before,away_wc_goals_for_before,away_wc_goals_against_before,away_wc_goal_diff_before,away_wc_wins_before,away_wc_win_rate_before,away_last3_matches,away_last3_points,away_last3_goals_for,away_last3_goals_against,away_last3_goal_diff,away_last3_win_rate,away_last5_matches,away_last5_points,away_last5_goals_for,away_last5_goals_against,away_last5_goal_diff,away_last5_win_rate,away_last10_matches,away_last10_points,away_last10_goals_for,away_last10_goals_against,away_last10_goal_diff,away_last10_win_rate,away_tournament_matches_before,away_tournament_points_before,away_tournament_goals_for_before,away_tournament_goals_against_before,away_tournament_goal_diff_before,away_rest_days,away_first_match_in_tournament,text_available,text_count,combined_pre_match_text,sources,latest_text_time,min_hours_before_match
480,2022,2022-12-02,Group stage,Korea Republic,Portugal,south korea,portugal,2,1,"Education City Stadium, Doha",Qatar,Facundo Tello,44097,Sérgio Costa,Fernando Santos,wc_2022,match_6eb13a82ec7c,team_3c6cfbc2415a,team_ae0eea6eb6d6,stadium_c69edb95a2c5,referee_a00e0d7d76d9,manager_b0e9b2e67a0f,manager_eff5489cb13e,home_win,0,28.0,1526.20,1519.54,6.66,2022-06-30,9.0,1678.65,1674.78,3.87,2022-06-30,-19.0,-152.45,2.79,False,36,28,36,73,-37,6,0.166667,3,4,4,3,1,0.333333,5,4,5,6,-1,0.2,10,6,11,16,-5,0.10,2,1,2,3,-1,4.0,False,32,54,54,37,17,16,0.500000,3,6,6,4,2,0.666667,5,10,8,5,3,0.6,10,15,15,16,-1,0.4,2,6,5,2,3,4.0,False,1,4,Canada 1-2 Morocco: World Cup 2022 – as it hap...,guardian,2022-12-01 17:31:08+00:00,6.481111
481,2022,2022-12-02,Group stage,Ghana,Uruguay,ghana,uruguay,0,2,"Al Janoub Stadium, Al Wakrah",Qatar,Daniel Siebert,43443,Otto Addo,Diego Alonso,wc_2022,match_8891748ec8e3,team_0e6049dbb2ce,team_9142ac5b229c,stadium_7d364161db5e,referee_2f871cd8fb56,manager_1c53f04ab922,manager_b7aa03c60b9a,away_win,2,60.0,1389.68,1387.36,2.32,2022-06-30,13.0,1643.71,1635.73,7.98,2022-06-30,-47.0,-254.03,-5.66,False,14,18,18,21,-3,5,0.357143,3,3,6,7,-1,0.333333,5,4,9,11,-2,0.2,10,12,14,15,-1,0.30,2,3,5,5,0,4.0,False,58,85,87,76,11,24,0.413793,3,1,0,4,-4,0.000000,5,7,5,5,0,0.4,10,19,10,8,2,0.6,2,1,0,2,-2,4.0,False,1,2,World Cup 2022 briefing: Iran and USA head int...,guardian,2022-11-29 04:00:16+00:00,67.995556
482,2022,2022-12-02,Group stage,Serbia,Switzerland,serbia,switzerland,2,3,"Stadium 974, Doha",Qatar,Fernando Rapallini,41378,Dragan Stojković,Murat Yakin,wc_2022,match_d71f633f94b5,team_a3867434e7dc,team_0d9c575f4f02,stadium_20634e82aefa,referee_0c4ba20de87e,manager_215cbb0d8230,manager_0a37ed79642a,away_win,2,25.0,1549.53,1547.53,2.00,2022-06-30,16.0,1621.43,1635.32,-13.8

# Сохранение датасета с текстами

Что делает: Сохраняет новый датасет:

In [140]:
save_table(
    modeling_dataset_text,
    PROCESSED_DIR / "match_dataset_1994_2022_with_text.parquet",
)

save_table(
    modeling_dataset_text,
    PROCESSED_DIR / "match_dataset_1994_2022_with_text.csv",
)


[OK] Saved parquet: /Users/gaperov/Documents/University/SNA/data/processed/match_dataset_1994_2022_with_text.parquet
[OK] Saved csv: /Users/gaperov/Documents/University/SNA/data/processed/match_dataset_1994_2022_with_text.csv


In [141]:
modeling_dataset_text.shape

(500, 109)